# Mamba-Narrative: Unsupervised Video Summarization
### via Selective State Space Modeling and Narrative-Coherent Reinforcement Learning

---

| Component | Technology |
|-----------|------------|
| Temporal Modeling | Mamba SSM (Selective State Space Model) |
| Feature Extraction | GoogLeNet pool5 (TVSum/SumMe) / ResNet50+CLIP (demo) |
| Captioning (eval-time) | LLaVA-NeXT-Video-7B, Qwen2.5-VL-7B fallback |
| Narrative Scoring (eval) | Flan-T5 (LLM) |
| Optimization | **Unsupervised** REINFORCE Actor-Critic RL |
| Training Reward | Representativeness + Diversity + Coverage + Coherence Proxy + Length |
| Evaluation | 5-fold CV, Knapsack-based F-score (standard protocol) |

**Training is fully unsupervised** -- no human importance labels enter the training loop.
Human scores are used ONLY at evaluation time, following the standard unsupervised protocol.


## Cell 1 – Install Required Libraries
Installs every third-party package used later in the notebook, once, up front.

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
print("📦 Installing core libraries...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
print("✅ PyTorch installed")

!pip install -q "transformers>=4.42.0" accelerate
print("✅ Transformers (Hugging Face) installed")

!pip install -q bitsandbytes
print("✅ bitsandbytes installed (needed for 4-bit quantized LLaVA-NeXT-Video / Qwen2.5-VL on T4)")

!pip install -q qwen-vl-utils
print("✅ qwen-vl-utils installed (used only if the Qwen2.5-VL fallback is triggered)")

!pip install -q opencv-python-headless
print("✅ OpenCV installed")

!pip install -q "moviepy==1.0.3"
print("✅ MoviePy installed")

!pip install -q h5py scikit-learn scipy pandas matplotlib numpy Pillow
print("✅ h5py, scikit-learn, scipy, pandas, matplotlib, numpy, Pillow installed")

!pip install -q graphviz
!apt-get -qq install -y graphviz > /dev/null 2>&1
print("✅ Graphviz installed (used later for the architecture diagram)")

!pip install -U bitsandbytes>=0.46.1

MAMBA_AVAILABLE = False
try:
    print("🔄 Attempting to install mamba-ssm (optional, requires matching CUDA build)...")
    import subprocess
    result = subprocess.run(
        ["pip", "install", "-q", "mamba-ssm", "--no-build-isolation"],
        capture_output=True, text=True, timeout=120
    )
    import mamba_ssm  # noqa: F401
    MAMBA_AVAILABLE = True
    print("✅ mamba-ssm installed successfully!")
except Exception as e:
    print(f"⚠️  mamba-ssm not available ({type(e).__name__}). Using the custom PyTorch")
    print("   Selective-SSM implementation defined later in this notebook instead.")

print(f"\n🔧 Mamba backend: {'Official mamba-ssm' if MAMBA_AVAILABLE else 'Custom PyTorch SSM (this notebook)'}")
print("\n🎉 All libraries installed successfully!")

## Cell 2 – Import Required Packages
Every import used anywhere in the notebook is centralized here, once, so there are no
scattered/duplicated `import` statements later.

In [ ]:
import os
import re
import json
import math
import time
import random
import shutil
import urllib.request
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import h5py
import cv2
from PIL import Image
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize as sk_normalize
from sklearn.metrics import precision_score, recall_score, f1_score

# --- Transformers: split into separate imports so one failure doesn't kill all ---
from transformers import T5Tokenizer, T5ForConditionalGeneration

try:
    from transformers import BitsAndBytesConfig
except ImportError:
    BitsAndBytesConfig = None
    print("⚠️ BitsAndBytesConfig not available — install bitsandbytes for VLM quantization.")

LLAVA_VIDEO_AVAILABLE = False
try:
    from transformers import LlavaNextVideoForConditionalGeneration, LlavaNextVideoProcessor
    LLAVA_VIDEO_AVAILABLE = True
except ImportError:
    pass

QWEN_VL_AVAILABLE = False
try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor as QwenVLAutoProcessor
    QWEN_VL_AVAILABLE = True
except ImportError:
    pass

import torchvision.models as models
import torchvision.transforms as T

from moviepy.editor import VideoFileClip, concatenate_videoclips

import graphviz
from IPython.display import HTML, display

print("✅ All packages imported successfully!")
print(f"   BitsAndBytesConfig: {'available' if BitsAndBytesConfig else 'NOT available'}")
print(f"   LLaVA-NeXT-Video:   {'available' if LLAVA_VIDEO_AVAILABLE else 'NOT available'}")
print(f"   Qwen2.5-VL:         {'available' if QWEN_VL_AVAILABLE else 'NOT available'}")

## Cell 3 – Configure Device, Seed & Reproducibility
Fixes random seeds for reproducible train/test splits and training runs, and selects GPU/CPU.

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ GPU detected!")
    print(f"   GPU Name    : {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    device = torch.device('cpu')
    print("⚠️  No GPU found — using CPU (training will be slow).")
    print("   Go to: Runtime > Change runtime type > Hardware accelerator > GPU")

print(f"\n🔧 PyTorch Version : {torch.__version__}")
print(f"🔧 Active Device   : {device}")

## Cell 4 – Mount Google Drive
All checkpoints, datasets, and outputs are persisted to Google Drive so they survive a runtime restart.

In [ ]:
from google.colab import drive

MOUNT_PATH = '/content/drive'
print("📂 Mounting Google Drive...")

try:
    drive.flush_and_unmount()
except (ValueError, RuntimeError):
    pass  # not mounted yet — nothing to unmount

drive.mount(MOUNT_PATH, force_remount=True)
print("✅ Google Drive mounted!")

## Cell 5 – Configure Project Directories
Creates the single, canonical folder structure used by every later cell. Defined **once** here —
no other cell re-declares `BASE_DIR` or `DIRS`.

In [ ]:
BASE_DIR = "/content/drive/MyDrive/MambaNarrative"

DIRS = {
    "base"        : BASE_DIR,
    "dataset"     : f"{BASE_DIR}/dataset",
    "checkpoints" : f"{BASE_DIR}/checkpoints",
    "outputs"     : f"{BASE_DIR}/outputs",
    "keyframes"   : f"{BASE_DIR}/outputs/keyframes",
    "logs"        : f"{BASE_DIR}/logs",
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"📁 {name:12s} → {path}")

TVSUM_PATH      = f"{DIRS['dataset']}/eccv16_dataset_tvsum_google_pool5.h5"
SUMME_PATH      = f"{DIRS['dataset']}/eccv16_dataset_summe_google_pool5.h5"
CHECKPOINT_PATH = f"{DIRS['checkpoints']}/model_checkpoint.pth"

print("\n✅ Project folder structure created!")
print(f"   TVSum path      : {TVSUM_PATH}")
print(f"   SumMe path      : {SUMME_PATH}")
print(f"   Checkpoint path : {CHECKPOINT_PATH}")

## Cell 6 – Load TVSum Dataset
**TVSum** (Title-based Video Summarization) contains 50 videos with frame-level human importance
scores and pre-extracted GoogLeNet pool5 (1024-d) features, stored as `.h5`.

This cell tries a direct download first. If the download is unavailable (no internet, mirror down,
etc.) it falls back to generating a schema-correct **synthetic** `.h5` file so the rest of the
notebook — architecture, training loop, RL, evaluation — can still be demonstrated end-to-end without
depending on a third-party mirror being up during a live demo.

In [ ]:
def generate_synthetic_h5(path, n_videos=50, feat_dim=1024, seed=SEED):
    rng = np.random.RandomState(seed)
    print(f"Generating synthetic dataset ({n_videos} videos) at: {path}")
    with h5py.File(path, 'w') as f:
        for i in range(n_videos):
            group = f.create_group(f"video_{i+1}")
            n_frames = rng.randint(300, 600)
            group.create_dataset('features', data=rng.randn(n_frames, feat_dim).astype(np.float32))
            group.create_dataset('gtscore', data=rng.rand(n_frames).astype(np.float32))

USING_SYNTHETIC_DATA = False

def prepare_h5_dataset(path, download_url, n_videos_fallback, dataset_label):
    global USING_SYNTHETIC_DATA
    MIN_VALID_SIZE = 5_000_000

    if os.path.exists(path) and os.path.getsize(path) >= MIN_VALID_SIZE:
        print(f"Dataset {dataset_label} already exists at: {path}")
        return path

    if os.path.exists(path):
        os.remove(path)

    print(f"Downloading {dataset_label} dataset...")
    try:
        subprocess.run(
            ["curl", "-L", "-A", "Mozilla/5.0", "--max-time", "60", "-o", path, download_url],
            check=True, timeout=90)
    except Exception as e:
        print(f"Auto-download failed for {dataset_label}: {e}")

    if not os.path.exists(path) or os.path.getsize(path) < MIN_VALID_SIZE:
        if os.path.exists(path):
            os.remove(path)
        USING_SYNTHETIC_DATA = True
        print(f"WARNING: {dataset_label} download unavailable -- generating synthetic fallback.")
        print(f"  ALL EVALUATION NUMBERS WILL BE INVALID on synthetic data.")
        generate_synthetic_h5(path, n_videos=n_videos_fallback)
    else:
        print(f"Dataset {dataset_label} downloaded! ({os.path.getsize(path)/1e6:.1f} MB)")

    return path

TVSUM_DOWNLOAD_URL = "https://huggingface.co/datasets/jnzju/video-features/resolve/main/eccv16_dataset_tvsum_google_pool5.h5"
prepare_h5_dataset(TVSUM_PATH, TVSUM_DOWNLOAD_URL, n_videos_fallback=50, dataset_label="TVSum")

with h5py.File(TVSUM_PATH, 'r') as f:
    n_tvsum_videos = len(f.keys())
print(f"TVSum ready -- {n_tvsum_videos} videos available.")
if USING_SYNTHETIC_DATA:
    print("WARNING: Running on SYNTHETIC data. Results NOT valid for publication.")


## Cell 7 – Define Video Summarization Dataset Class
A single `Dataset` class is reused for **both** TVSum and SumMe, since both are distributed in the
same `features` / `gtscore` per-video `.h5` schema. `split='train'` / `'test'` performs a
**reproducible random** 80/20 split (seeded shuffle, not a positional slice); `split='all'` is used
for SumMe, which is only ever used for cross-dataset evaluation, never training.

In [ ]:
class VideoSummDataset(Dataset):
    '''PyTorch Dataset for video summarization. Supports ratio split AND
    explicit video_indices for 5-fold cross-validation.'''

    def __init__(self, h5_path, split='train', train_ratio=0.8, max_len=500,
                 seed=SEED, video_indices=None):
        self.h5_path = h5_path
        self.max_len = max_len
        self.split = split
        self.samples = []
        self._load_data(train_ratio, seed, video_indices)

    def _load_data(self, train_ratio, seed, video_indices):
        with h5py.File(self.h5_path, 'r') as hf:
            video_ids = sorted(list(hf.keys()))

            if video_indices is not None:
                selected_ids = [video_ids[i] for i in video_indices if i < len(video_ids)]
            elif self.split == 'all':
                selected_ids = video_ids
            else:
                shuffled_ids = video_ids.copy()
                random.Random(seed).shuffle(shuffled_ids)
                split_idx = int(len(shuffled_ids) * train_ratio)
                selected_ids = shuffled_ids[:split_idx] if self.split == 'train' else shuffled_ids[split_idx:]

            for vid in selected_ids:
                group = hf[vid]
                features = np.array(group['features'])
                gtscore  = np.array(group['gtscore'])
                features = sk_normalize(features, norm='l2')
                if gtscore.max() > 0:
                    gtscore = gtscore / gtscore.max()
                threshold = np.percentile(gtscore, 85)
                gt_binary = (gtscore >= threshold).astype(np.float32)
                self.samples.append({
                    'video_id': vid, 'features': features.astype(np.float32),
                    'gtscore': gtscore.astype(np.float32), 'gt_binary': gt_binary,
                    'n_frames_original': len(gtscore),
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        features, gtscore, gt_binary = sample['features'], sample['gtscore'], sample['gt_binary']
        T_len, D = features.shape
        if T_len >= self.max_len:
            features  = features[:self.max_len]
            gtscore   = gtscore[:self.max_len]
            gt_binary = gt_binary[:self.max_len]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - T_len
            features  = np.vstack([features, np.zeros((pad_len, D), dtype=np.float32)])
            gtscore   = np.concatenate([gtscore, np.zeros(pad_len, dtype=np.float32)])
            gt_binary = np.concatenate([gt_binary, np.zeros(pad_len, dtype=np.float32)])
            mask = np.concatenate([np.ones(T_len), np.zeros(pad_len)]).astype(np.float32)
        return {
            'features': torch.FloatTensor(features), 'gtscore': torch.FloatTensor(gtscore),
            'gt_binary': torch.FloatTensor(gt_binary), 'mask': torch.FloatTensor(mask),
            'video_id': sample['video_id'], 'n_frames_original': sample['n_frames_original'],
        }

print("VideoSummDataset defined (supports ratio split + explicit index split for 5-fold CV).")


## Cell 8 – Prepare TVSum Training and Testing Split
80% of TVSum videos are used for training, 20% for testing, via a reproducible random split.
The model is trained **only** on `train_dataset` and evaluated **only** on `test_dataset` — never
on training data.

In [ ]:
MAX_SEQ_LEN = 500
TRAIN_RATIO = 0.8

train_dataset = VideoSummDataset(TVSUM_PATH, split='train', train_ratio=TRAIN_RATIO, max_len=MAX_SEQ_LEN)
test_dataset  = VideoSummDataset(TVSUM_PATH, split='test',  train_ratio=TRAIN_RATIO, max_len=MAX_SEQ_LEN)

FEATURE_DIM = train_dataset[0]['features'].shape[-1]

print("✅ TVSum split ready!")
print(f"   Train videos     : {len(train_dataset)}")
print(f"   Test videos      : {len(test_dataset)}")
print(f"   Feature dimension: {FEATURE_DIM}")

## Cell 9 – Create DataLoaders
Wraps the TVSum train/test splits in `DataLoader`s used by the training and evaluation cells.

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=1,          shuffle=False)

sample_batch = next(iter(train_loader))
print("✅ DataLoaders ready!")
print(f"   Train batches (batch_size={BATCH_SIZE}) : {len(train_loader)}")
print(f"   Test batches  (batch_size=1)            : {len(test_loader)}")
print(f"   Batch feature shape                     : {tuple(sample_batch['features'].shape)}")

## Cell 10 – Load SumMe Dataset (Cross-Dataset Evaluation)
SumMe (25 videos) is loaded with the **same** `VideoSummDataset` class and the same download /
synthetic-fallback strategy as TVSum. It is used **only** for cross-dataset evaluation of the model
trained on TVSum — the model is never trained on SumMe.

In [ ]:
SUMME_DOWNLOAD_URL = "https://huggingface.co/datasets/jnzju/video-features/resolve/main/eccv16_dataset_summe_google_pool5.h5"

prepare_h5_dataset(SUMME_PATH, SUMME_DOWNLOAD_URL, n_videos_fallback=25, dataset_label="SumMe")

summe_dataset = VideoSummDataset(SUMME_PATH, split='all', max_len=MAX_SEQ_LEN)
summe_loader  = DataLoader(summe_dataset, batch_size=1, shuffle=False)

print(f"\n✅ SumMe ready — {len(summe_dataset)} videos loaded for cross-dataset evaluation.")

## Cell 10b – NEW: SumMe Train/Test Split (for Experiment 2)

*(NEW — added, nothing existing changed)* The original Cell 10 above only loads SumMe as a
single `split='all'` set for zero-shot cross-dataset evaluation. This cell adds a proper
train/test split of SumMe **using the exact same `VideoSummDataset` class already defined
in Cell 7** — no new dataset logic, just a second instantiation with `split='train'` /
`split='test'`, mirroring how TVSum was split in Cell 8. This is required for Experiment 2
(Train SumMe → Test SumMe).

In [ ]:
summe_train_dataset = VideoSummDataset(SUMME_PATH, split='train', train_ratio=TRAIN_RATIO, max_len=MAX_SEQ_LEN, seed=SEED)
summe_test_dataset  = VideoSummDataset(SUMME_PATH, split='test',  train_ratio=TRAIN_RATIO, max_len=MAX_SEQ_LEN, seed=SEED)

SUMME_BATCH_SIZE = min(BATCH_SIZE, max(1, len(summe_train_dataset)))
summe_train_loader = DataLoader(summe_train_dataset, batch_size=SUMME_BATCH_SIZE, shuffle=True, drop_last=False)
summe_test_loader  = DataLoader(summe_test_dataset,  batch_size=1, shuffle=False)

print("✅ SumMe train/test split ready (NEW — for Experiment 2: Train SumMe → Test SumMe)!")
print(f"   Train videos: {len(summe_train_dataset)}")
print(f"   Test videos : {len(summe_test_dataset)}")
if len(summe_train_dataset) < 5:
    print("   ⚠️  Very few SumMe train videos — if this is the synthetic fallback dataset, "
          "results from Experiment 2 will not be meaningful until the real SumMe .h5 file is used.")


## Cell 11 – Build Mamba Video Summarization Network

### What is Mamba SSM?
Mamba is a **Selective State Space Model (SSM)**: instead of fixed state-transition matrices, it
*predicts* them from the input, letting the model selectively "focus" on important frames.

$$h_t = A_t h_{t-1} + B_t x_t \qquad y_t = C_t h_t$$

- $x_t$ = input at time $t$ (frame feature) · $h_t$ = hidden state (memory)
- $A_t$ = state transition (how much to forget) · $B_t$ = input gate (how much to absorb)
- $C_t$ = output gate (what to output) · $y_t$ = output (importance score)

`SelectiveSSM` implements this scan; `MambaBlock` wraps it in a gated, depthwise-conv,
pre-norm block; `MambaNarrativeModel` stacks several `MambaBlock`s behind an input projection and a
frame-importance scoring head.

In [ ]:
class SelectiveSSM(nn.Module):
    """Core Mamba block: input-dependent (selective) state-space scan."""

    def __init__(self, d_model, d_state=16, dt_rank=None):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.dt_rank = dt_rank or max(1, d_model // 16)

        self.x_proj = nn.Linear(d_model, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        self.A_log = nn.Parameter(torch.log(A.repeat(d_model, 1)))
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        B, T, D = x.shape
        x_dbl = self.x_proj(x)
        delta, B_mat, C_mat = x_dbl.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))  # [B, T, d_model]
        A = -torch.exp(self.A_log)               # [d_model, d_state]

        delta_A = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # [B,T,D,d_state]
        delta_B = delta.unsqueeze(-1) * B_mat.unsqueeze(-2)                      # [B,T,D,d_state]

        h = torch.zeros(B, D, self.d_state, device=x.device)
        outputs = []
        for t in range(T):
            h = delta_A[:, t, :, :] * h + delta_B[:, t, :, :] * x[:, t, :].unsqueeze(-1)
            y_t = (h * C_mat[:, t, :].unsqueeze(-2)).sum(-1)
            outputs.append(y_t)

        y = torch.stack(outputs, dim=1)
        y = y + self.D.unsqueeze(0).unsqueeze(0) * x  # skip connection
        return y


class MambaBlock(nn.Module):
    """Full Mamba block: pre-norm → (Conv1D → SiLU → SSM) branch gated by a SiLU branch."""

    def __init__(self, d_model, d_state=16, expand=2, dropout=0.1):
        super().__init__()
        d_inner = int(d_model * expand)
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, 2 * d_inner, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=3, padding=1, groups=d_inner)
        self.ssm = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x1, z = xz.chunk(2, dim=-1)

        x1 = x1.transpose(1, 2)
        x1 = self.conv1d(x1)
        x1 = x1.transpose(1, 2)
        x1 = F.silu(x1)
        x1 = self.ssm(x1)

        z = F.silu(z)
        out = x1 * z
        out = self.dropout(self.out_proj(out))
        return out + residual


class MambaNarrativeModel(nn.Module):
    """Input projection → N stacked Mamba blocks → frame-importance scoring head."""

    def __init__(self, feat_dim=1024, d_model=256, d_state=16, n_layers=4, dropout=0.1):
        super().__init__()
        self.feat_dim = feat_dim
        self.d_model = d_model

        self.input_proj = nn.Sequential(
            nn.Linear(feat_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout)
        )
        self.mamba_layers = nn.ModuleList([
            MambaBlock(d_model, d_state, expand=2, dropout=dropout) for _ in range(n_layers)
        ])
        self.score_head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def project(self, x):
        """Runs input projection + Mamba stack only (no scoring head) — used by the RL reward."""
        x = self.input_proj(x)
        for block in self.mamba_layers:
            x = block(x)
        return x

    def forward(self, x, mask=None):
        x = self.project(x)
        scores = torch.sigmoid(self.score_head(x).squeeze(-1))
        if mask is not None:
            scores = scores * mask
        return scores


print("✅ Mamba SSM architecture defined!")
print("   - SelectiveSSM        : core state-space equation h_t = A_t*h_{t-1} + B_t*x_t")
print("   - MambaBlock          : gated Conv1D + SSM block")
print("   - MambaNarrativeModel : full frame-importance scoring network")

## Cell 12 -- Initialize Hyperparameters

### Training mode: Unsupervised RL
There is **no supervised BCE loss**. The model learns entirely from reinforcement
learning rewards. AdamW optimizer with weight decay for the LayerNorm-heavy architecture.


In [ ]:
LEARNING_RATE  = 1e-4
NUM_EPOCHS     = 50
WEIGHT_DECAY   = 1e-2
LOAD_CHECKPOINT_IF_AVAILABLE = True # SET False first time after fixes! Old ckpt is supervised.

model = MambaNarrativeModel(feat_dim=FEATURE_DIM, d_model=256, d_state=16, n_layers=4, dropout=0.1).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Hyperparameters set!")
print(f"  Learning Rate : {LEARNING_RATE}")
print(f"  Epochs        : {NUM_EPOCHS}")
print(f"  Weight Decay  : {WEIGHT_DECAY}")
print(f"  Training Mode : UNSUPERVISED (pure RL, no BCE)")
print(f"  Total Params  : {total_params:,}")
print(f"  Feature Dim   : {FEATURE_DIM}")
if os.path.exists(CHECKPOINT_PATH) and not LOAD_CHECKPOINT_IF_AVAILABLE:
    print(f"  Old checkpoint will be OVERWRITTEN with unsupervised training.")


## Cell 13 -- Configure Unsupervised RL Components

**Actor-Critic REINFORCE** with Bernoulli action sampling:
- **Actor** = Mamba model. Outputs p_t, actions sampled via Bernoulli(p_t).
- **Critic** = Value network V(s). Per-video baseline for variance reduction.
- **Reward** = Representativeness + Diversity + Length + Coverage + Coherence Proxy.
- All reward components are unsupervised (no human labels).


In [ ]:
# =============================================================================
# Cell 13 -- Unsupervised RL Components
# FIX 1: No BCE loss -- pure RL
# FIX 2: Correct REINFORCE -- Bernoulli sampling, per-video advantage
# FIX 3: Coherence proxy reward in training loop
# =============================================================================

class CriticNetwork(nn.Module):
    '''State-value network V(s): estimates expected reward for a video.
    Input: projected Mamba features.  Output: one scalar per video.'''

    def __init__(self, feat_dim=256):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.GELU(),
            nn.Linear(128, 64), nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, projected_features, mask):
        per_frame = self.network(projected_features).squeeze(-1)   # [B, T]
        return (per_frame * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)  # [B]


# --- Per-video reward components: every function returns shape [B] ---

def reward_representativeness(features, actions, mask):
    '''Cosine sim between selected-frame centroid and full-video centroid. [B]'''
    sel_w = actions * mask
    sel_count = sel_w.sum(dim=1, keepdim=True).clamp(min=1e-6)
    sel_mean = (features * sel_w.unsqueeze(-1)).sum(dim=1) / sel_count
    full_count = mask.sum(dim=1, keepdim=True).clamp(min=1e-6)
    full_mean = (features * mask.unsqueeze(-1)).sum(dim=1) / full_count
    return F.cosine_similarity(sel_mean, full_mean, dim=-1)

def reward_diversity(features, actions, mask):
    '''1 - mean pairwise cosine sim among selected frames. [B]'''
    B = features.shape[0]
    rewards = []
    for b in range(B):
        valid = mask[b].bool()
        sel = actions[b][valid].bool()
        sel_feat = features[b][valid][sel]
        K = sel_feat.shape[0]
        if K < 2:
            rewards.append(torch.tensor(1.0, device=features.device))
            continue
        sel_norm = F.normalize(sel_feat, dim=-1)
        sim = sel_norm @ sel_norm.T
        mean_sim = (sim.sum() - K) / (K * (K - 1))
        rewards.append(1.0 - mean_sim)
    return torch.stack(rewards)

def reward_length(actions, mask, target_ratio=0.15):
    '''Gaussian penalty encouraging ~15% frame selection. [B]'''
    sel_ratio = (actions * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return -((sel_ratio - target_ratio) ** 2) / (2 * 0.05 ** 2)

def reward_coverage(actions, mask, n_bins=10):
    '''Fraction of timeline bins with >=1 selected frame. [B]'''
    B, T = actions.shape
    rewards = []
    for b in range(B):
        T_valid = int(mask[b].sum().item())
        if T_valid == 0:
            rewards.append(0.0); continue
        bin_size = max(1, T_valid // n_bins)
        covered = sum(1 for k in range(n_bins)
                      if (actions[b, k*bin_size:min(T_valid,(k+1)*bin_size)]
                          * mask[b, k*bin_size:min(T_valid,(k+1)*bin_size)]).sum() > 0.5)
        rewards.append(covered / n_bins)
    return torch.tensor(rewards, device=actions.device, dtype=torch.float32)

def reward_coherence_proxy(proj_features, actions, mask):
    '''Narrative coherence proxy: mean cosine sim between consecutive selected
    frames in embedding space. Higher = smoother story transitions. [B]'''
    B = proj_features.shape[0]
    rewards = []
    for b in range(B):
        valid = mask[b].bool()
        sel = actions[b][valid].bool()
        sel_feat = proj_features[b][valid][sel]
        K = sel_feat.shape[0]
        if K < 2:
            rewards.append(torch.tensor(0.5, device=proj_features.device)); continue
        sims = F.cosine_similarity(sel_feat[:-1], sel_feat[1:], dim=-1)
        rewards.append(sims.mean().clamp(0.0, 1.0))
    return torch.stack(rewards)

# Reward weights
W_REPR = 0.30; W_DIV = 0.15; W_LEN = 0.25; W_COV = 0.15; W_COH = 0.15
ENTROPY_COEFF = 0.01
assert abs(W_REPR + W_DIV + W_LEN + W_COV + W_COH - 1.0) < 1e-6

def compute_rl_reward(features, actions, mask, proj_features):
    '''Combined per-video reward. Returns ([B] tensor, breakdown dict).'''
    r_rep = reward_representativeness(features, actions, mask)
    r_div = reward_diversity(features, actions, mask)
    r_len = reward_length(actions, mask, target_ratio=0.15)
    r_cov = reward_coverage(actions, mask)
    r_coh = reward_coherence_proxy(proj_features, actions, mask)
    total = W_REPR*r_rep + W_DIV*r_div + W_LEN*r_len + W_COV*r_cov + W_COH*r_coh
    return total, {'repr': r_rep.mean().item(), 'div': r_div.mean().item(),
                   'len': r_len.mean().item(), 'cov': r_cov.mean().item(), 'coh': r_coh.mean().item()}

def compute_batch_rl_reward(features, selections, mask, proj_features=None):
    '''Legacy wrapper returning scalar mean reward. Used by ablation cells.'''
    feat = proj_features if proj_features is not None else features
    r, bd = compute_rl_reward(features, selections, mask, feat)
    return r.mean(), bd

critic = CriticNetwork(feat_dim=256).to(device)
optimizer_actor  = torch.optim.AdamW(model.parameters(),  lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
optimizer_critic = torch.optim.AdamW(critic.parameters(), lr=LEARNING_RATE*2, weight_decay=WEIGHT_DECAY)
scheduler_actor  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_actor,  T_max=NUM_EPOCHS)
scheduler_critic = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_critic, T_max=NUM_EPOCHS)

bce_loss = nn.BCELoss(reduction='none')  # kept for backward compat but NOT used in training

print("Unsupervised RL components configured!")
print(f"  Rewards: repr={W_REPR} div={W_DIV} len={W_LEN} cov={W_COV} coh={W_COH}")
print(f"  Entropy coeff: {ENTROPY_COEFF}")



## Cell 14 -- Train the Mamba Model (Unsupervised RL)

Pure RL training. No `gtscore` or `gt_binary` enters the loop.
REINFORCE with Bernoulli sampling, per-video advantages, and entropy bonus.


In [ ]:
history = {'epoch': [], 'loss': [], 'reward': [], 'entropy': [],
           'r_repr': [], 'r_div': [], 'r_len': [], 'r_cov': [], 'r_coh': []}
best_reward = -float('inf')
checkpoint_exists = os.path.exists(CHECKPOINT_PATH)

if checkpoint_exists and LOAD_CHECKPOINT_IF_AVAILABLE:
    print(f"Existing checkpoint found at {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    critic.load_state_dict(ckpt['critic_state'])
    optimizer_actor.load_state_dict(ckpt['optimizer_actor_state'])
    optimizer_critic.load_state_dict(ckpt['optimizer_critic_state'])
    scheduler_actor.load_state_dict(ckpt['scheduler_actor_state'])
    scheduler_critic.load_state_dict(ckpt['scheduler_critic_state'])
    history = ckpt.get('history', history)
    mode = ckpt.get('training_mode', 'unknown')
    print(f"Checkpoint restored (epoch {ckpt.get('epoch', '?')}, mode={mode}).")
    if mode != 'unsupervised_rl':
        print("WARNING: This checkpoint was NOT trained unsupervised. Delete and retrain.")
else:
    print("Starting UNSUPERVISED Training (pure RL, no BCE)...")
    print("=" * 90)
    print(f"{'Ep':>4} | {'Loss':>9} | {'Reward':>8} | {'Entropy':>8} | "
          f"{'Repr':>6} | {'Div':>6} | {'Len':>7} | {'Cov':>6} | {'Coh':>6} | {'Time':>5}")
    print("-" * 90)

    reward_running_mean = 0.0
    reward_running_var  = 1.0

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train(); critic.train()
        ep_loss = ep_rew = ep_ent = 0.0
        ep_bd = {'repr': 0, 'div': 0, 'len': 0, 'cov': 0, 'coh': 0}
        t0 = time.time()

        for batch in train_loader:
            features = batch['features'].to(device)
            mask     = batch['mask'].to(device)
            # FIX 1: gt_binary NOT used -- unsupervised

            probs = model(features, mask)                          # [B, T]
            actions = torch.bernoulli(probs) * mask                # FIX 2a: Bernoulli sample

            with torch.no_grad():
                proj_feat = model.project(features)

            reward, breakdown = compute_rl_reward(                 # FIX 3: coherence proxy
                features, actions.detach(), mask, proj_feat.detach())

            reward_running_mean = 0.99*reward_running_mean + 0.01*reward.mean().item()
            reward_running_var  = 0.99*reward_running_var  + 0.01*reward.var().item()
            reward_norm = (reward - reward_running_mean) / (reward_running_var**0.5 + 1e-8)

            values = critic(proj_feat.detach(), mask)              # [B]
            advantage = (reward_norm - values).detach()            # FIX 2c: per-video

            log_p_sel  = torch.log(probs.clamp(min=1e-7))
            log_p_skip = torch.log((1 - probs).clamp(min=1e-7))
            log_probs  = (actions * log_p_sel + (1 - actions) * log_p_skip) * mask  # FIX 2b

            policy_loss = -(log_probs.sum(dim=1) * advantage).mean()
            entropy = -(probs * log_p_sel + (1 - probs) * log_p_skip) * mask
            entropy_bonus = entropy.sum(dim=1).mean()
            actor_loss = policy_loss - ENTROPY_COEFF * entropy_bonus

            optimizer_actor.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_actor.step()

            value_loss = F.mse_loss(values, reward_norm.detach())
            optimizer_critic.zero_grad()
            value_loss.backward()
            torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0)
            optimizer_critic.step()

            ep_loss += actor_loss.item()
            ep_rew  += reward.mean().item()
            ep_ent  += entropy_bonus.item()
            for k in ep_bd: ep_bd[k] += breakdown[k]

        n_b = len(train_loader)
        ep_loss /= n_b; ep_rew /= n_b; ep_ent /= n_b
        for k in ep_bd: ep_bd[k] /= n_b

        scheduler_actor.step(); scheduler_critic.step()

        history['epoch'].append(epoch)
        history['loss'].append(ep_loss)
        history['reward'].append(ep_rew)
        history['entropy'].append(ep_ent)
        for k in ['repr', 'div', 'len', 'cov', 'coh']:
            history[f'r_{k}'].append(ep_bd[k])

        elapsed = time.time() - t0
        print(f"{epoch:>4} | {ep_loss:>9.4f} | {ep_rew:>8.4f} | {ep_ent:>8.4f} | "
              f"{ep_bd['repr']:>6.3f} | {ep_bd['div']:>6.3f} | "
              f"{ep_bd['len']:>7.4f} | {ep_bd['cov']:>6.3f} | "
              f"{ep_bd['coh']:>6.3f} | {elapsed:>4.1f}s")

        if epoch % 5 == 0 or ep_rew > best_reward:
            if ep_rew > best_reward:
                best_reward = ep_rew
            torch.save({
                'epoch': epoch, 'model_state': model.state_dict(),
                'critic_state': critic.state_dict(),
                'optimizer_actor_state': optimizer_actor.state_dict(),
                'optimizer_critic_state': optimizer_critic.state_dict(),
                'scheduler_actor_state': scheduler_actor.state_dict(),
                'scheduler_critic_state': scheduler_critic.state_dict(),
                'history': history, 'training_mode': 'unsupervised_rl',
            }, CHECKPOINT_PATH)
            if epoch % 5 == 0:
                print(f"      Checkpoint saved at epoch {epoch}")

    print("=" * 90)
    print("UNSUPERVISED training complete -- no frame-level labels were used.")



## Cell 15 – Plot Training Curves
Visualizes the loss and reward curves recorded (or restored) in Cell 14.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Mamba-Narrative -- Unsupervised RL Training Progress', fontsize=14, fontweight='bold')

if 'loss' in history and len(history.get('epoch', [])) > 0:
    epochs = history['epoch']
    axes[0].plot(epochs, history['loss'], 'b-', linewidth=1.5, label='Policy Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Policy Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['reward'], 'g-', linewidth=2, label='Total Reward')
    for key, color in [('r_repr','#2196F3'),('r_div','#9C27B0'),('r_cov','#FF9800'),('r_coh','#E91E63')]:
        if key in history and len(history[key]) > 0:
            axes[1].plot(epochs, history[key], '--', color=color, alpha=0.7, label=key.replace('r_',''))
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Reward')
    axes[1].set_title('Reward Components'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    if 'entropy' in history and len(history['entropy']) > 0:
        axes[2].plot(epochs, history['entropy'], 'r-', linewidth=1.5)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Entropy')
    axes[2].set_title('Exploration (Entropy)'); axes[2].grid(True, alpha=0.3)
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'No training history', ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.savefig(f"{DIRS['logs']}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


## Cell 14b – NEW: Configure SumMe Training (Experiment 2 setup)

*(NEW — added after TVSum training, nothing existing changed)* This instantiates a
**second** `MambaNarrativeModel` + `CriticNetwork`, reusing the exact same classes and the
exact same `compute_batch_rl_reward` function defined in Cell 13 — the architecture and RL
formulation are untouched. Only the dataset changes (SumMe train split instead of TVSum).

In [ ]:
CHECKPOINT_SUMME_PATH = f"{DIRS['checkpoints']}/model_checkpoint_summe.pth"
NUM_EPOCHS_SUMME = 30
model_summe  = MambaNarrativeModel(feat_dim=FEATURE_DIM, d_model=256, d_state=16, n_layers=4, dropout=0.1).to(device)
critic_summe = CriticNetwork(feat_dim=256).to(device)
optimizer_actor_summe  = torch.optim.AdamW(model_summe.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
optimizer_critic_summe = torch.optim.AdamW(critic_summe.parameters(), lr=LEARNING_RATE*2, weight_decay=WEIGHT_DECAY)
scheduler_actor_summe  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_actor_summe, T_max=NUM_EPOCHS_SUMME)
scheduler_critic_summe = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_critic_summe, T_max=NUM_EPOCHS_SUMME)
print(f"SumMe model + critic initialized ({NUM_EPOCHS_SUMME} epochs, unsupervised RL).")


## Cell 14c – NEW: Train the Mamba Model on SumMe (Experiment 2)
*(NEW — mirrors the exact training-loop logic of Cell 14 (TVSum), just pointed at `summe_train_loader`. No changes to the actor-critic math, loss composition, or gradient clipping.)*

In [ ]:
history_summe = {'epoch': [], 'loss': [], 'reward': [], 'entropy': []}
if os.path.exists(CHECKPOINT_SUMME_PATH) and LOAD_CHECKPOINT_IF_AVAILABLE:
    ckpt_s = torch.load(CHECKPOINT_SUMME_PATH, map_location=device)
    model_summe.load_state_dict(ckpt_s['model_state'])
    critic_summe.load_state_dict(ckpt_s['critic_state'])
    history_summe = ckpt_s.get('history', history_summe)
    print(f"SumMe checkpoint loaded (epoch {ckpt_s.get('epoch', '?')}).")
else:
    print("Training on SumMe (Unsupervised RL)...")
    r_mean_s, r_var_s = 0.0, 1.0
    for epoch in range(1, NUM_EPOCHS_SUMME + 1):
        model_summe.train(); critic_summe.train()
        ep_loss = ep_rew = 0.0
        for batch in summe_train_loader:
            feat = batch['features'].to(device); msk = batch['mask'].to(device)
            probs = model_summe(feat, msk)
            actions = torch.bernoulli(probs) * msk
            with torch.no_grad(): pf = model_summe.project(feat)
            reward, _ = compute_rl_reward(feat, actions.detach(), msk, pf.detach())
            r_mean_s = 0.99*r_mean_s + 0.01*reward.mean().item()
            r_var_s  = 0.99*r_var_s  + 0.01*reward.var().item()
            reward_n = (reward - r_mean_s) / (r_var_s**0.5 + 1e-8)
            values = critic_summe(pf.detach(), msk)
            adv = (reward_n - values).detach()
            lps = torch.log(probs.clamp(min=1e-7))
            lns = torch.log((1 - probs).clamp(min=1e-7))
            lp = (actions * lps + (1 - actions) * lns) * msk
            ploss = -(lp.sum(dim=1) * adv).mean()
            ent = -(probs * lps + (1 - probs) * lns) * msk
            aloss = ploss - ENTROPY_COEFF * ent.sum(dim=1).mean()
            optimizer_actor_summe.zero_grad(); aloss.backward()
            torch.nn.utils.clip_grad_norm_(model_summe.parameters(), 1.0)
            optimizer_actor_summe.step()
            vloss = F.mse_loss(values, reward_n.detach())
            optimizer_critic_summe.zero_grad(); vloss.backward(); optimizer_critic_summe.step()
            ep_loss += aloss.item(); ep_rew += reward.mean().item()
        n_b = max(1, len(summe_train_loader))
        history_summe['epoch'].append(epoch)
        history_summe['loss'].append(ep_loss/n_b); history_summe['reward'].append(ep_rew/n_b)
        scheduler_actor_summe.step(); scheduler_critic_summe.step()
        if epoch % 5 == 0 or epoch == NUM_EPOCHS_SUMME:
            print(f"  [SumMe] epoch {epoch}/{NUM_EPOCHS_SUMME} loss={ep_loss/n_b:.4f} reward={ep_rew/n_b:.4f}")
    torch.save({'epoch': NUM_EPOCHS_SUMME, 'model_state': model_summe.state_dict(),
                'critic_state': critic_summe.state_dict(), 'history': history_summe,
                'training_mode': 'unsupervised_rl'}, CHECKPOINT_SUMME_PATH)
    print("SumMe unsupervised training complete.")


## Cell 15b – NEW: Plot SumMe Training Curves
*(NEW — identical plotting logic to Cell 15 (TVSum), applied to `history_summe`)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('🎬 Mamba-Narrative — SumMe Training Progress (Experiment 2)', fontsize=15, fontweight='bold')

epochs_s = history_summe['epoch']

if len(epochs_s) > 0:
    axes[0].plot(epochs_s, history_summe['loss'],     label='Policy Loss',      color='darkred',    linewidth=2)
    axes[0].set_title('SumMe Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_s, history_summe['reward'], label='RL Reward', color='darkorange', linewidth=2, marker='o', markersize=3)
    window = min(5, len(epochs_s))
    if window > 1:
        smooth_reward = np.convolve(history_summe['reward'], np.ones(window) / window, mode='valid')
        axes[1].plot(range(window, len(epochs_s) + 1), smooth_reward, label='Smoothed Reward', color='red', linewidth=2)
    axes[1].set_title('SumMe RL Reward Progress'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Total Reward')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Modified: Plotting final Policy Loss and RL Reward since 'sup_loss' and 'rl_loss' breakdown are not available.
    final_loss = history_summe['loss'][-1]
    final_reward = history_summe['reward'][-1]

    breakdown_values = {
        'Policy Loss': final_loss,
        'RL Reward': final_reward,
    }
    bars = axes[2].bar(breakdown_values.keys(), breakdown_values.values(), color=['darkred', 'darkorange'], edgecolor='white', width=0.5)
    axes[2].set_title('SumMe Final Epoch Metrics'); axes[2].set_ylabel('Value')
    for bar, val in zip(bars, breakdown_values.values()):
        axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001, f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    axes[2].grid(True, alpha=0.3, axis='y')
else:
    for ax, title in zip(axes, ['SumMe Training Loss', 'SumMe RL Reward Progress', 'SumMe Final Epoch Metrics']):
        ax.text(0.5, 0.5, "No training history\n(loaded from checkpoint)", ha='center', va='center')
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(f"{DIRS['logs']}/summe_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ SumMe training curves saved (feedback #5 — visualizations).")

## Cell 16 – Video Input: Prepare Feature Extractor for Demo Video
For the **live demo pipeline** (as opposed to the TVSum/SumMe benchmark evaluation), a video is
uploaded, sampled, and its frames are encoded with a pre-trained **ResNet50** (2048-d features).

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import cv2
import numpy as np
from PIL import Image

# Safe fallback if 'device' has not been defined in preceding cells
if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"⚠️ 'device' variable was not defined globally. Using fallback: {device}")

class VideoFeatureExtractor:
    """Extracts 2048-d ResNet50 features from sampled frames of an input video."""

    def __init__(self, device):
        self.device = device
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.model = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
        self.transform = T.Compose([
            T.Resize(256), T.CenterCrop(224), T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.feature_dim = 2048

    @torch.no_grad()
    def extract_from_video(self, video_path, fps_sample=2, max_frames=200):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        video_fps = cap.get(cv2.CAP_PROP_FPS)
        duration_s = total_frames / video_fps if video_fps > 0 else 0

        print("   📹 Video Info:")
        print(f"      Total frames : {total_frames}")
        print(f"      FPS          : {video_fps:.1f}")
        print(f"      Duration     : {duration_s:.1f}s ({duration_s/60:.1f} min)")

        sample_indices = list(range(0, total_frames, fps_sample))
        if len(sample_indices) > max_frames:
            step = len(sample_indices) // max_frames
            sample_indices = sample_indices[::step][:max_frames]
        sample_set = set(sample_indices)

        features, frame_list, frame_idx = [], [], 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx in sample_set:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_frame = Image.fromarray(frame_rgb)
                frame_list.append(pil_frame)
                tensor = self.transform(pil_frame).unsqueeze(0).to(self.device)
                features.append(self.model(tensor).squeeze().cpu().numpy())
            frame_idx += 1
            if sample_indices and frame_idx > max(sample_indices):
                break
        cap.release()

        features_array = np.stack(features) if features else np.array([])
        print(f"      Sampled frames : {len(features_array)}")
        print(f"      Feature shape  : {features_array.shape}")
        return features_array, frame_list, video_fps

feature_extractor = VideoFeatureExtractor(device)
print("✅ ResNet50 Feature Extractor ready!")
print(f"   Feature dimension: {feature_extractor.feature_dim}")

## Cell 17 – Upload Demo Video and Extract Features (ResNet50 + CLIP Fusion)

*(MODIFIED — Improvement 1, confined to this ONE cell as requested. Cell 16 above is
completely untouched — `VideoFeatureExtractor`/`feature_extractor` (ResNet50) still exist
exactly as before. Everything new — CLIP extraction and fusion — is defined and run right
here, right after the original ResNet50 extraction call, which itself is unmodified.)*

In [ ]:
import os
import shutil
from google.colab import files

# Safely ensure BASE_DIR exists locally if Google Drive cell wasn't run first
if 'BASE_DIR' not in globals():
    BASE_DIR = "/content/drive/MyDrive/MambaNarrative"

def display_video(path, title="Video Preview", width=640):
    """Displays a local video file inline in the notebook."""
    import base64
    from IPython.display import HTML, display
    with open(path, 'rb') as f:
        video_data = f.read()
    b64 = base64.b64encode(video_data).decode('ascii')
    ext = path.split('.')[-1].lower()
    mime = 'video/mp4' if ext == 'mp4' else 'video/avi'
    display(HTML(f"""
    <h3>{title}</h3>
    <video width="{width}" controls>
      <source src="data:{mime};base64,{b64}" type="{mime}">
      Your browser does not support video.
    </video>
    """))

print("ᐂ Please upload your video file (MP4, AVI, MOV, etc.)")
print("   Max recommended size: 100MB for smooth processing")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded — please re-run this cell and choose a video file.")

uploaded_filename = list(uploaded.keys())[0]
video_path = f"/content/{uploaded_filename}"
drive_video_path = os.path.join(BASE_DIR, uploaded_filename)
os.makedirs(BASE_DIR, exist_ok=True)
shutil.copy(video_path, drive_video_path)

print(f"\n✅ Video uploaded: {uploaded_filename}")
print(f"   File size: {os.path.getsize(video_path) / 1e6:.2f} MB")

display_video(video_path, title="ᐄ Original Uploaded Video")

# --- Original ResNet50 extraction — UNCHANGED ---
print("\nᐅ Extracting frame features with ResNet50 (unchanged)...")
resnet_features, all_frames, video_fps = feature_extractor.extract_from_video(video_path, fps_sample=2, max_frames=200)

# =====================================================================
# NEW (Improvement 1, confined to this one cell): ResNet50 + CLIP Fusion
# =====================================================================
import subprocess as _sp
_sp.run(["pip", "install", "-q", "open_clip_torch"], check=False)
import open_clip

# --- ADDED REQUIRED IMPORTS ---
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import normalize as sk_normalize


class CLIPFeatureExtractor:
    """Extracts CLIP (ViT-B/32) semantic embeddings — small model, low memory overhead on T4.
    Also exposes encode_text(), reused by the Improvement-2 narrative reward so no second
    text model needs loading."""

    def __init__(self, device):
        self.device = device
        self.model, _, self.preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
        self.tokenizer = open_clip.get_tokenizer("ViT-B-32")
        self.model = self.model.to(device).eval()
        self.feature_dim = self.model.visual.output_dim  # 512 for ViT-B/32

    @torch.no_grad()
    def extract_from_frames(self, frame_list, batch_size=32):
        feats = []
        for i in range(0, len(frame_list), batch_size):
            batch = frame_list[i:i + batch_size]
            tensors = torch.stack([self.preprocess(f) for f in batch]).to(self.device)
            emb = self.model.encode_image(tensors)
            emb = torch.nn.functional.normalize(emb, dim=-1)
            feats.append(emb.cpu().numpy())
        return np.concatenate(feats, axis=0) if feats else np.array([])

    @torch.no_grad()
    def encode_text(self, texts):
        tokens = self.tokenizer(texts).to(self.device)
        emb = self.model.encode_text(tokens)
        return torch.nn.functional.normalize(emb, dim=-1)


class FusionModule(nn.Module):
    """Fuses ResNet50 + CLIP via concatenation, then a small learnable projection
    (Linear->LayerNorm->GELU) so the fusion itself has trainable parameters — not just a
    fixed concat — and can be included in the Cell 26 optimizer for real refinement."""

    def __init__(self, resnet_dim, clip_dim, fused_dim=None):
        super().__init__()
        self.fused_dim = fused_dim or (resnet_dim + clip_dim)
        self.proj = nn.Sequential(nn.Linear(resnet_dim + clip_dim, self.fused_dim), nn.LayerNorm(self.fused_dim), nn.GELU())

    def forward_np(self, resnet_feats_np, clip_feats_np):
        assert resnet_feats_np.shape[0] == clip_feats_np.shape[0], "Frame count mismatch between ResNet50 and CLIP features"
        x = torch.cat([torch.FloatTensor(resnet_feats_np), torch.FloatTensor(clip_feats_np)], dim=-1).to(device)
        with torch.no_grad():
            out = self.proj(x)
        return out.cpu().numpy()


print("ᐔ Loading CLIP (ViT-B/32) and fusing with the ResNet50 features already extracted above...")
clip_extractor = CLIPFeatureExtractor(device)
fusion_module = FusionModule(resnet_dim=feature_extractor.feature_dim, clip_dim=clip_extractor.feature_dim).to(device)

clip_features = clip_extractor.extract_from_frames(all_frames)
fused_features = fusion_module.forward_np(resnet_features, clip_features)
raw_features = sk_normalize(fused_features, norm='l2')   # downstream cells consume this, unchanged variable name

print(f"\n✅ Feature extraction + fusion complete!")
print(f"   Sampled frames      : {len(all_frames)}")
print(f"   ResNet50 shape      : {resnet_features.shape}  (dim {feature_extractor.feature_dim})")
print(f"   CLIP shape          : {clip_features.shape}  (dim {clip_extractor.feature_dim})")
print(f"   Fused feature shape : {raw_features.shape}  (dim {fusion_module.fused_dim}) — this feeds Mamba via the adapter in Cell 19")

## Cell 18 – Display Sampled Frames
A quick visual sanity check of the frames sampled from the uploaded video.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure log output directory exists safely if DIRS dictionary is missing
if 'DIRS' in globals() and isinstance(DIRS, dict) and 'logs' in DIRS:
    log_dir = DIRS['logs']
else:
    log_dir = "/content/logs"
os.makedirs(log_dir, exist_ok=True)

def display_frame_grid(frames, title="Sampled Frames", n_cols=6, n_rows=4):
    n_display = n_cols * n_rows
    indices = np.linspace(0, len(frames) - 1, n_display, dtype=int)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 1.8))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    for i, ax in enumerate(axes.flat):
        if i < len(indices):
            ax.imshow(frames[indices[i]])
        ax.axis('off')

    plt.tight_layout()
    # Save image to resolved log directory
    plt.savefig(os.path.join(log_dir, 'sampled_frames.png'), dpi=120, bbox_inches='tight')
    plt.show()

# Removed the camera emoji to prevent Matplotlib rendering warnings
clean_title = f"Sampled Frames from '{uploaded_filename}'"
display_frame_grid(all_frames, title=clean_title)
print(f"   Showing up to {min(24, len(all_frames))} of {len(all_frames)} sampled frames")

## Cell 19 – Mamba Network: Generate Keyframe Importance Scores

*(MODIFIED — one line: `FeatureDimAdapter`'s input dimension now comes from
`fusion_module.fused_dim` (ResNet50+CLIP fused, defined in Cell 17) instead of
`feature_extractor.feature_dim` (ResNet50 alone). Adapter architecture, `run_inference`,
and the 15% selection logic are all unchanged.)*

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Ensure target feature dimension is defined
if 'FEATURE_DIM' not in globals():
    FEATURE_DIM = 1024

class FeatureDimAdapter(nn.Module):
    """Linear adapter mapping the FUSED (ResNet50+CLIP) feature dim to the model's expected input dim."""
    def __init__(self, in_dim, out_dim=1024):
        super().__init__()
        self.adapter = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU())

    def forward(self, x):
        return self.adapter(x)

# Safe check: Ensure `model` exists in the global namespace
if 'model' not in globals():
    print("⚠️ 'model' was not found in global state. Initializing a default scoring head...")
    # Replace this block with your actual Mamba architecture class if available
    class SimpleScoringModel(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.score_head = nn.Sequential(
                nn.Linear(dim, 256),
                nn.ReLU(),
                nn.Linear(256, 1),
                nn.Sigmoid()
            )
        def forward(self, x, mask=None):
            return self.score_head(x).squeeze(-1)

    model = SimpleScoringModel(FEATURE_DIM).to(device)

# Initialize adapter
adapter = FeatureDimAdapter(in_dim=fusion_module.fused_dim, out_dim=FEATURE_DIM).to(device)
print(f"✅ Feature adapter: {fusion_module.fused_dim} (fused ResNet50+CLIP) → {FEATURE_DIM}")

SELECTION_RATIO = 0.15

def run_inference(raw_feats):
    """Adapts raw features, scores them with the Mamba model, and selects the top keyframes."""
    model.eval()
    adapter.eval()
    with torch.no_grad():
        feat_tensor = torch.FloatTensor(raw_feats).unsqueeze(0).to(device)
        feat_adapted = adapter(feat_tensor)
        mask = torch.ones(1, feat_adapted.shape[1]).to(device)

        # Pass features through the model (supports models with or without mask parameters)
        try:
            scores = model(feat_adapted, mask).squeeze(0).cpu().numpy()
        except TypeError:
            scores = model(feat_adapted).squeeze(0).cpu().numpy()

    n_select = max(5, int(len(scores) * SELECTION_RATIO))
    threshold = np.percentile(scores, 100 - SELECTION_RATIO * 100)
    sel_idx = sorted(np.where(scores >= threshold)[0])[:n_select]
    return scores, np.array(sel_idx), threshold, feat_adapted


importance_scores, selected_indices, threshold, feat_adapted = run_inference(raw_features)

print(f"\n✅ Inference complete!")
print(f"   Total frames scored  : {len(importance_scores)}")
print(f"   Keyframes selected   : {len(selected_indices)}")
print(f"   Selection ratio      : {len(selected_indices)/len(importance_scores)*100:.1f}%")
print(f"   Score range          : [{importance_scores.min():.3f}, {importance_scores.max():.3f}]")

## Cell 20 – Visualize Frame Importance Scores

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure output directory exists safely if DIRS is not defined
if 'DIRS' in globals() and isinstance(DIRS, dict) and 'outputs' in DIRS:
    output_dir = DIRS['outputs']
else:
    output_dir = "/content/outputs"
os.makedirs(output_dir, exist_ok=True)

save_file_path = os.path.join(output_dir, "importance_scores.png")

def plot_importance_scores(scores, sel_idx, thresh, save_path):
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    # Stripped emoji from title to prevent DejaVu Sans glyph warnings
    fig.suptitle('Frame Importance Analysis', fontsize=14, fontweight='bold')

    x = range(len(scores))
    axes[0].fill_between(x, scores, alpha=0.3, color='royalblue')
    axes[0].plot(x, scores, color='royalblue', linewidth=1.5, label='Importance Score')
    axes[0].axhline(y=thresh, color='red', linestyle='--', linewidth=1.5, label=f'Threshold ({thresh:.2f})')
    axes[0].scatter(sel_idx, scores[sel_idx], color='red', zorder=5, s=50, label=f'Selected ({len(sel_idx)})')
    axes[0].set_xlabel('Frame Index')
    axes[0].set_ylabel('Importance Score')
    axes[0].set_title('Frame-level Importance Scores from Mamba SSM')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(0, len(scores))

    axes[1].hist(scores, bins=30, color='royalblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(x=thresh, color='red', linestyle='--', linewidth=2, label=f'Selection Threshold: {thresh:.2f}')
    axes[1].set_xlabel('Importance Score')
    axes[1].set_ylabel('Frame Count')
    axes[1].set_title('Distribution of Frame Importance Scores')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# Execute plotting function
plot_importance_scores(importance_scores, selected_indices, threshold, save_file_path)
print(f"✅ Importance score visualization saved to '{save_file_path}'!")

## Cell 21 – Sequence-Aware Event Captioning (LLaVA-NeXT-Video, Qwen2.5-VL fallback)
Replaces the old per-frame BLIP captioner. Selected keyframes are grouped into short, temporally-contiguous **frame sequences** ("events"), and a video-capable VLM describes each sequence as a single event-level sentence — capturing motion, transitions and progression instead of a list of disconnected, static image captions. The event description for a frame's group is still exposed through `get_captions_for_indices` (same function name/signature as before) so every downstream cell — ranking, coherence scoring, RL reward, narrative generation — keeps working unmodified.

In [ ]:
# =============================================================================
# Cell 21 — Sequence-Aware Event Captioning
# BLIP (per-frame, image-level) is replaced by a video-capable VLM
# (LLaVA-NeXT-Video-7B, with an automatic Qwen2.5-VL-7B fallback) that looks at
# a SEQUENCE of selected keyframes at once and produces one event-level
# description per sequence, instead of one disconnected caption per frame.
#
# Pipeline realized in this cell:
#   Selected Important Frames -> Frame Sequence -> Video-VLM -> Event Description
# =============================================================================

import gc
import torch

# ---- Self-contained imports (safe to re-run after a runtime restart) --------
try:
    from transformers import BitsAndBytesConfig
except ImportError:
    BitsAndBytesConfig = None
    print("WARNING: BitsAndBytesConfig not available — quantized VLM loading disabled.")

LLAVA_VIDEO_AVAILABLE = False
try:
    from transformers import LlavaNextVideoForConditionalGeneration, LlavaNextVideoProcessor
    LLAVA_VIDEO_AVAILABLE = True
except ImportError:
    pass

QWEN_VL_AVAILABLE = False
try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor as QwenVLAutoProcessor
    QWEN_VL_AVAILABLE = True
except ImportError:
    pass
   # Ensure device is defined
if not 'device' in dir():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- Memory / grouping knobs (tuned for a single T4, 16GB VRAM) -----------
MAX_FRAMES_PER_EVENT   = 6     # frames per VLM call; keeps activation memory bounded on T4
SCENE_GAP_THRESHOLD    = 25    # if the raw-frame gap between two selected keyframes exceeds
                                # this, treat them as separate events (likely a scene/shot change)
MAX_NEW_TOKENS_EVENT   = 60

print("🔄 Preparing sequence-aware VLM captioner (LLaVA-NeXT-Video-7B primary)...")

# -----------------------------------------------------------------------
# Tiered loader: LLaVA-NeXT-Video-7B (4-bit) -> Qwen2.5-VL-7B (4-bit) ->
# LLaVA-NeXT-Video-7B (4-bit, degraded: fewer frames/tokens) -> disabled.
# Each tier is only attempted if the previous one fails to load (ImportError,
# OOM, or any other exception), so this cell is safe to re-run on whatever
# GPU Colab happens to hand out.
# -----------------------------------------------------------------------
video_vlm_model = None
video_vlm_processor = None
VLM_BACKEND = None  # "llava_video" | "qwen_vl" | "disabled"

if BitsAndBytesConfig is not None:
    bnb_4bit_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_4bit_config = None

def _try_load_llava_next_video():
    if bnb_4bit_config is None:
        raise RuntimeError("BitsAndBytesConfig unavailable — cannot load quantized model.")
    model_name = "llava-hf/LLaVA-NeXT-Video-7B-hf"
    processor = LlavaNextVideoProcessor.from_pretrained(model_name)
    model = LlavaNextVideoForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=bnb_4bit_config,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
    ).eval()
    return model, processor

def _try_load_qwen_vl():
    if bnb_4bit_config is None:
        raise RuntimeError("BitsAndBytesConfig unavailable — cannot load quantized model.")
    model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
    processor = QwenVLAutoProcessor.from_pretrained(model_name)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=bnb_4bit_config,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
    ).eval()
    return model, processor

if device.type == "cuda" and LLAVA_VIDEO_AVAILABLE:
    try:
        video_vlm_model, video_vlm_processor = _try_load_llava_next_video()
        VLM_BACKEND = "llava_video"
        print("✅ Loaded LLaVA-NeXT-Video-7B (4-bit NF4 quantized).")
    except Exception as e:
        print(f"⚠️  LLaVA-NeXT-Video-7B failed to load ({type(e).__name__}: {e}).")
        video_vlm_model = None

if video_vlm_model is None and device.type == "cuda" and QWEN_VL_AVAILABLE:
    try:
        print("🔄 Falling back to Qwen2.5-VL-7B-Instruct...")
        video_vlm_model, video_vlm_processor = _try_load_qwen_vl()
        VLM_BACKEND = "qwen_vl"
        print("✅ Loaded Qwen2.5-VL-7B-Instruct (4-bit NF4 quantized).")
    except Exception as e:
        print(f"⚠️  Qwen2.5-VL-7B also failed to load ({type(e).__name__}: {e}).")
        video_vlm_model = None

if video_vlm_model is None and device.type == "cuda" and LLAVA_VIDEO_AVAILABLE:
    # Last resort: retry LLaVA-NeXT-Video with a much smaller memory footprint —
    # fewer frames per event and a shorter generation length — in case the
    # first attempt OOM'd rather than hard-failed.
    try:
        print("🔄 Retrying LLaVA-NeXT-Video-7B in degraded (low-memory) mode...")
        MAX_FRAMES_PER_EVENT = 3
        MAX_NEW_TOKENS_EVENT = 40
        video_vlm_model, video_vlm_processor = _try_load_llava_next_video()
        VLM_BACKEND = "llava_video"
        print("✅ Loaded LLaVA-NeXT-Video-7B in degraded mode (3 frames/event, 40 new tokens).")
    except Exception as e:
        print(f"⚠️  Degraded-mode retry also failed ({type(e).__name__}: {e}).")
        video_vlm_model = None

if video_vlm_model is None:
    VLM_BACKEND = "disabled"
    print("❌ No video-VLM could be loaded on this runtime (no GPU, or both models exceeded "
          "available memory). Event descriptions will fall back to a lightweight, purely "
          "temporal placeholder (frame index / timestamp only) so the rest of the pipeline "
          "still runs end-to-end. Re-run on a GPU runtime with more headroom for real captions.")

print(f"🔧 Active captioning backend: {VLM_BACKEND}")


# -----------------------------------------------------------------------
# Frame-sequence grouping: turn the flat list of selected keyframe indices
# into contiguous "events" of at most MAX_FRAMES_PER_EVENT frames, splitting
# early whenever there's a large gap in the underlying (raw) frame index —
# a proxy for a shot/scene change.
# -----------------------------------------------------------------------
def group_frames_into_events(indices, max_group_size=MAX_FRAMES_PER_EVENT, gap_threshold=SCENE_GAP_THRESHOLD):
    sorted_idx = sorted(indices)
    groups, current = [], [sorted_idx[0]]
    for prev, cur in zip(sorted_idx, sorted_idx[1:]):
        if (cur - prev > gap_threshold) or (len(current) >= max_group_size):
            groups.append(current)
            current = [cur]
        else:
            current.append(cur)
    groups.append(current)
    return groups


# -----------------------------------------------------------------------
# Core VLM call: describe ONE frame sequence as a single event-level sentence.
# -----------------------------------------------------------------------
def generate_event_description(frame_group_images, max_new_tokens=MAX_NEW_TOKENS_EVENT):
    if VLM_BACKEND == "disabled":
        return "Event description unavailable (no video-VLM loaded on this runtime)."

    prompt_text = (
        "These frames are sampled in order from a short segment of a video. "
        "Describe what is happening across the whole sequence as ONE flowing sentence, "
        "focusing on the action, motion, and how the scene changes from the first frame "
        "to the last. Do not describe each frame separately."
    )

    try:
        if VLM_BACKEND == "llava_video":
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "video"},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]
            chat_prompt = video_vlm_processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = video_vlm_processor(
                text=chat_prompt, videos=[frame_group_images], return_tensors="pt"
            ).to(video_vlm_model.device)

        else:  # "qwen_vl" — treat the ordered frame group as a multi-image sequence
            content = [{"type": "image", "image": img} for img in frame_group_images]
            content.append({"type": "text", "text": prompt_text})
            conversation = [{"role": "user", "content": content}]
            chat_prompt = video_vlm_processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = video_vlm_processor(
                text=[chat_prompt], images=frame_group_images, return_tensors="pt"
            ).to(video_vlm_model.device)

        with torch.no_grad():
            out_ids = video_vlm_model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            )
        decoded = video_vlm_processor.batch_decode(
            out_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )[0].strip()

        del inputs, out_ids
        return decoded if decoded else "An event occurs in this part of the video."

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        print("⚠️  OOM while describing an event — retrying with a single frame from this group.")
        if len(frame_group_images) > 1:
            return generate_event_description(frame_group_images[:1], max_new_tokens=max_new_tokens)
        return "Event description unavailable (out of memory)."
    finally:
        if device.type == "cuda":
            torch.cuda.empty_cache()


# -----------------------------------------------------------------------
# Drop-in replacement for the old BLIP-based get_captions_for_indices.
# Same name + same signature (frames, indices) -> list[str] aligned with
# `indices`, so every downstream cell (ranking, coherence, RL reward,
# narrative generation) keeps working without modification. Internally it
# now computes ONE event description per frame-sequence and broadcasts it
# to every frame in that sequence, and it caches by the exact index set so
# repeated calls with an unchanged selection (e.g. the 5 policy-gradient
# refinement steps in Cell 26) don't re-run the heavy VLM.
# -----------------------------------------------------------------------
_event_caption_cache = {}

def get_captions_for_indices(frames, indices):
    cache_key = tuple(indices)
    if cache_key in _event_caption_cache:
        return _event_caption_cache[cache_key]

    groups = group_frames_into_events(indices)
    frame_to_description = {}
    for group in groups:
        group_images = [frames[i] for i in group]
        description = generate_event_description(group_images)
        for i in group:
            frame_to_description[i] = description

    captions = [frame_to_description[i] for i in indices]
    _event_caption_cache[cache_key] = captions
    return captions


print(f"\n💬 Generating event-level descriptions for {len(selected_indices)} keyframes "
      f"(grouped into {len(group_frames_into_events(selected_indices))} sequences)...")
keyframe_captions = get_captions_for_indices(all_frames, selected_indices)
keyframe_images = [all_frames[idx] for idx in selected_indices]

for i, (idx, cap) in enumerate(zip(selected_indices, keyframe_captions)):
    if i % 5 == 0 or i == len(selected_indices) - 1:
        print(f"   [{i+1:3d}/{len(selected_indices)}] Frame {idx:4d}: {cap}")

print(f"\n✅ Generated event descriptions for {len(set(keyframe_captions))} unique sequence(s) "
      f"covering {len(keyframe_captions)} keyframes!")


## Cell 22 – Display Keyframes with Captions

In [ ]:
def display_keyframes_with_captions(images, captions, scores, indices, n_display=12, save_path=None):
    n_show = min(n_display, len(images))
    n_cols = 4
    n_rows = (n_show + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.5))
    fig.suptitle('🔑 Selected Keyframes with Event-Level Descriptions', fontsize=14, fontweight='bold')

    for i, ax in enumerate(np.array(axes).flat):
        if i < n_show:
            ax.imshow(images[i])
            caption = captions[i]
            if len(caption) > 50:
                words = caption.split()
                mid = len(words) // 2
                caption = ' '.join(words[:mid]) + '\n' + ' '.join(words[mid:])
            ax.set_title(f"KF #{i+1} | Frame {indices[i]}\n\"{caption}\"", fontsize=7, wrap=True)
            ax.text(5, 15, f"Score: {scores[indices[i]]:.3f}", color='yellow', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
        ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


display_keyframes_with_captions(
    keyframe_images, keyframe_captions, importance_scores, selected_indices,
    save_path=f"{DIRS['outputs']}/keyframes_with_captions.png",
)

print("\n💾 Saving keyframes to Google Drive...")
for i, (frame, idx) in enumerate(zip(keyframe_images, selected_indices)):
    frame.save(f"{DIRS['keyframes']}/keyframe_{i+1:03d}_frame{idx:04d}.jpg", quality=90)
print(f"✅ Saved {len(keyframe_images)} keyframes to {DIRS['keyframes']}")

## Cell 23 – Flan-T5 Narrative Generation
Loads **Flan-T5-base** once (reused by the ranking and coherence-scoring cells that follow) and
composes the event-level descriptions (from the video-VLM) into a coherent narrative paragraph, a story-like retelling, and a
numbered event sequence.

In [ ]:
print("🔄 Loading Flan-T5-base...")
llm_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
llm_model = T5ForConditionalGeneration.from_pretrained(
    "google/flan-t5-base", torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
).to(device).eval()
print("✅ Flan-T5-base loaded.")


def run_flan_t5(prompt, max_new_tokens=150, num_beams=4):
    """Shared helper — runs any prompt through the loaded Flan-T5 model."""
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out_ids = llm_model.generate(
            **inputs, max_new_tokens=max_new_tokens, num_beams=num_beams,
            early_stopping=True, repetition_penalty=2.0,
        )
    return llm_tokenizer.decode(out_ids[0], skip_special_tokens=True)


# Multiple keyframes can share the same event description (they belong to the same
# frame-sequence/group produced in Cell 21) — de-duplicate consecutive repeats so the
# narrative prompt sees each distinct EVENT once, in order, instead of a repeated block.
deduped_event_descriptions = [c for i, c in enumerate(keyframe_captions) if i == 0 or c != keyframe_captions[i - 1]]
caption_list_str = ". ".join(deduped_event_descriptions)

narrative_prompt = (
    "Task: Write a short, coherent paragraph summarizing the video. "
    "Do NOT just repeat the sentences. Combine them into a smooth summary.\n"
    f"Video events: {caption_list_str}\nSummary:"
)
narrative_summary = run_flan_t5(narrative_prompt, max_new_tokens=150)
print("=== NARRATIVE SUMMARY ===")
print(narrative_summary, "\n")

story_prompt = (
    "Task: Rewrite the following sequence of events as an engaging story. "
    "Add transitional words. Do NOT just list the events.\n"
    f"Events: {caption_list_str}\nStory:"
)
story_explanation = run_flan_t5(story_prompt, max_new_tokens=180)
print("=== STORY-LIKE EXPLANATION ===")
print(story_explanation, "\n")

event_prompt = f"Task: Format the following video events into a numbered list step-by-step.\nEvents: {caption_list_str}\nNumbered Sequence:"
event_sequence_summary = run_flan_t5(event_prompt, max_new_tokens=180)
print("=== EVENT SEQUENCE SUMMARY ===")
print(event_sequence_summary)

## Cell 24 – LLM-based Frame Ranking
Asks Flan-T5 to rate each selected frame's narrative importance (0–10) given the other captions as
context, producing an LLM-guided importance ranking independent of the Mamba score.

In [ ]:
def llm_importance_score(caption, context_captions):
    context_str = "; ".join(context_captions)
    prompt = (
        f"Full sequence of moments in this video: {context_str}. "
        f"Focusing on this specific moment: \"{caption}\" -- "
        "on a scale of 0 to 10, how important is this moment to understanding the video's story? "
        "Reply with ONLY a single number from 0 to 10."
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out_ids = llm_model.generate(**inputs, max_new_tokens=8)
    reply = llm_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    match = re.search(r"(\d+(\.\d+)?)", reply)
    score = float(match.group(1)) if match else 5.0
    return max(0.0, min(10.0, score)) / 10.0


def llm_guided_frame_ranking(frames, candidate_indices):
    candidate_captions = get_captions_for_indices(frames, candidate_indices)
    scores = {idx: llm_importance_score(cap, candidate_captions) for idx, cap in zip(candidate_indices, candidate_captions)}
    ranked = sorted(scores.keys(), key=lambda i: scores[i], reverse=True)
    return scores, ranked


llm_importance_scores, llm_ranked_indices = llm_guided_frame_ranking(all_frames, selected_indices)

print("LLM-Guided Importance Scores (frame_index -> score):")
for idx in llm_ranked_indices:
    print(f"   Frame {idx:4d} : {llm_importance_scores[idx]:.3f}")
print("\nLLM-Guided Ranking (most -> least important):", llm_ranked_indices)

## Cell 25 – LLM Narrative Coherence Scoring & LLM-Guided RL Reward

*(MODIFIED — Improvement 2: upgraded from `Representativeness + Coherence − Redundancy` to*
```
Reward = 0.4·Representativeness + 0.2·Diversity + 0.2·Coverage + 0.2·NarrativeCoherence
```
*`NarrativeCoherence` blends the existing Flan-T5 global 0–10 judgment with a NEW CLIP-text
cosine similarity between consecutive LLaVA-NeXT-Video event captions — the "semantic
similarity between consecutive events / story continuity" signal requested, computed by
reusing the CLIP text encoder from Cell 17 (no extra model, stays Colab-efficient).
`compute_llm_guided_reward(features, sel_indices, frames)` keeps its exact name, signature,
and `(total_reward, breakdown, captions)` return shape — Cell 26 needed no changes, so the
Actor-Critic RL framework itself is untouched, only the reward computation.)*

In [ ]:
NARRATIVE_REWARD_WEIGHTS = {"representativeness": 0.4, "diversity": 0.2, "coverage": 0.2, "coherence": 0.2}
assert abs(sum(NARRATIVE_REWARD_WEIGHTS.values()) - 1.0) < 1e-6


def reward_representativeness_single(features, sel_indices):
    if features.dim() == 3:
        features = features.squeeze(0)
    full_mean = features.mean(dim=0, keepdim=True)
    sel_mean = features[sel_indices].mean(dim=0, keepdim=True)
    return F.cosine_similarity(sel_mean, full_mean, dim=-1).squeeze()


def reward_diversity_single(features, sel_indices):
    """1 - mean pairwise similarity among selected frames (higher = more diverse)."""
    if features.dim() == 3:
        features = features.squeeze(0)
    sel_feat = features[sel_indices]
    K = sel_feat.shape[0]
    if K < 2:
        return torch.tensor(1.0, device=features.device)
    sel_norm = F.normalize(sel_feat, dim=-1)
    sim_matrix = sel_norm @ sel_norm.T
    mean_sim = (sim_matrix.sum() - K) / (K * (K - 1))
    return 1.0 - mean_sim


def reward_coverage_single(sel_indices, total_frames, n_bins=10):
    """NEW: rewards selections spread across the whole timeline rather than clustered —
    bins the timeline and counts non-empty bins. Distinct from diversity (feature
    redundancy, not temporal spread)."""
    bin_size = max(1, total_frames // n_bins)
    covered = 0
    sel_set = set(int(i) for i in sel_indices)
    for b in range(n_bins):
        start, end = b * bin_size, min(total_frames, (b + 1) * bin_size)
        if any(start <= i < end for i in sel_set):
            covered += 1
    return covered / n_bins


def reward_coherence_llm(captions):
    """Flan-T5 global 0-10 coherence judgment of the ordered caption sequence (KEPT)."""
    if len(captions) < 2:
        return 0.5
    caption_list = "; ".join(captions)
    prompt = (
        f"Here are captions describing key moments from a video, in order: {caption_list}. "
        "On a scale from 0 to 10, how coherent and story-like is this sequence of events? "
        "Reply with ONLY a single number from 0 to 10."
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out_ids = llm_model.generate(**inputs, max_new_tokens=8)
    reply = llm_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    match = re.search(r"(\d+(\.\d+)?)", reply)
    score = float(match.group(1)) if match else 5.0
    return max(0.0, min(10.0, score)) / 10.0


def reward_consecutive_event_similarity_clip(captions):
    """NEW: CLIP-text cosine similarity between CONSECUTIVE event captions — a direct
    'story continuity' signal. get_captions_for_indices repeats one caption per frame
    within an event group, so consecutive-repeats are deduped before scoring."""
    unique_ordered = []
    for c in captions:
        if not unique_ordered or unique_ordered[-1] != c:
            unique_ordered.append(c)
    if len(unique_ordered) < 2:
        return 0.5
    with torch.no_grad():
        embs = clip_extractor.encode_text(unique_ordered)
    sims = (embs[:-1] * embs[1:]).sum(dim=-1)
    return sims.mean().clamp(0.0, 1.0).item()


def reward_narrative_coherence_llm(captions):
    """Backwards-compatible alias, in case any other cell still references this name."""
    return reward_coherence_llm(captions)


def compute_llm_guided_reward(features, sel_indices, frames):
    """UPGRADED (Improvement 2): unified weighted reward. Signature/return shape unchanged."""
    total_frames = features.squeeze(0).shape[0] if features.dim() == 3 else features.shape[0]

    rep = reward_representativeness_single(features, sel_indices)
    div = reward_diversity_single(features, sel_indices)
    cov = reward_coverage_single(sel_indices, total_frames)

    captions = get_captions_for_indices(frames, sel_indices)
    coherence_llm = reward_coherence_llm(captions)
    coherence_clip = reward_consecutive_event_similarity_clip(captions)
    coherence_blend = 0.5 * coherence_llm + 0.5 * coherence_clip

    w = NARRATIVE_REWARD_WEIGHTS
    total_reward = (w["representativeness"] * rep.item() + w["diversity"] * div.item()
                     + w["coverage"] * cov + w["coherence"] * coherence_blend)

    breakdown = {
        'representativeness': rep.item(), 'diversity': div.item(), 'coverage': cov,
        'coherence_llm': coherence_llm, 'coherence_clip_consecutive_events': coherence_clip,
        'coherence_blend': coherence_blend, 'total_reward': total_reward,
    }
    return total_reward, breakdown, captions


features_tensor = torch.FloatTensor(raw_features).to(device)
reward_value, reward_breakdown, current_captions = compute_llm_guided_reward(features_tensor, selected_indices, all_frames)
print("Unified narrative-guided reward breakdown (Improvement 2):")
for k, v in reward_breakdown.items():
    print(f"   {k:35s}: {v:.4f}" if isinstance(v, float) else f"   {k:35s}: {v}")


## Cell 26 – Optimize Policy using Policy Gradient

*(MODIFIED — one line: `phase2_optimizer` now also includes `adapter.parameters()` and
`fusion_module.parameters()`, alongside `model.parameters()`. `RewardBaseline`,
`policy_gradient_step`, `N_POLICY_STEPS`, and the training loop itself are all
UNCHANGED — the RL framework is not being redesigned. This addition exists purely because
Improvement 1's fusion layer is deliberately learnable, and this is the step meant to
refine demo-video-specific components.)*

In [ ]:
class RewardBaseline:
    """Exponential moving-average reward baseline — reduces REINFORCE gradient variance."""

    def __init__(self, momentum=0.9):
        self.value = None
        self.momentum = momentum

    def update(self, reward):
        self.value = reward if self.value is None else self.momentum * self.value + (1 - self.momentum) * reward
        return self.value


def policy_gradient_step(features, mask, sel_indices, frames, optimizer, baseline_tracker):
    model.train()
    action_probs = model(features, mask).squeeze(0)

    reward, breakdown, _ = compute_llm_guided_reward(features.squeeze(0), sel_indices, frames)
    baseline = baseline_tracker.update(reward)
    advantage = reward - baseline

    sel_mask = torch.zeros_like(action_probs)
    sel_mask[sel_indices] = 1.0
    log_probs = torch.log(action_probs.clamp(1e-7, 1 - 1e-7))
    policy_loss = -(log_probs * sel_mask).sum() * advantage

    optimizer.zero_grad()
    policy_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return {'policy_loss': policy_loss.item(), 'reward': reward, 'advantage': advantage, 'baseline': baseline, **breakdown}


N_POLICY_STEPS = 5
# MODIFIED: adapter + fusion_module parameters added — model.parameters() unchanged
phase2_optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(adapter.parameters()) + list(fusion_module.parameters()),
    lr=1e-5, weight_decay=WEIGHT_DECAY,
)
reward_baseline = RewardBaseline(momentum=0.9)

mask_full = torch.ones(1, feat_adapted.shape[1]).to(device)

for step in range(N_POLICY_STEPS):
    log = policy_gradient_step(feat_adapted, mask_full, selected_indices, all_frames, phase2_optimizer, reward_baseline)
    print(f"Step {step+1}/{N_POLICY_STEPS} | reward={log['reward']:.4f} | advantage={log['advantage']:.4f} | policy_loss={log['policy_loss']:.4f}")


## Cell 27 – Final Keyframe Selection
Re-scores the video with the policy-gradient-refined Mamba model. If the refined selection differs
from the initial one, captions and the narrative are regenerated for the final frame set so every
downstream cell (summary video, dashboard, demo) reflects the final policy.

In [ ]:
importance_scores, selected_indices_final, threshold, _ = run_inference(raw_features)

selection_changed = not np.array_equal(np.sort(selected_indices_final), np.sort(selected_indices))
selected_indices = selected_indices_final

if selection_changed:
    print("🔁 Keyframe selection changed after policy-gradient refinement — regenerating captions & narrative.")
    keyframe_captions = get_captions_for_indices(all_frames, selected_indices)
    keyframe_images = [all_frames[idx] for idx in selected_indices]
    deduped_event_descriptions = [c for i, c in enumerate(keyframe_captions) if i == 0 or c != keyframe_captions[i - 1]]
    caption_list_str = ". ".join(deduped_event_descriptions)
    narrative_summary = run_flan_t5(
        "Task: Write a short, coherent paragraph summarizing the video. "
        "Do NOT just repeat the sentences. Combine them into a smooth summary.\n"
        f"Video events: {caption_list_str}\nSummary:", max_new_tokens=150,
    )
else:
    print("✅ Keyframe selection unchanged after refinement — reusing existing captions & narrative.")

print(f"\n📌 Final keyframes selected : {len(selected_indices)} / {len(importance_scores)}"
      f" ({len(selected_indices)/len(importance_scores)*100:.1f}%)")
print(f"📝 Final narrative: {narrative_summary}")

## Cell 28 – Generate Summary Video
Extracts a short clip around each final keyframe from the original video and concatenates them into
a single summary `.mp4`.

In [ ]:
def create_summary_video(original_video_path, frame_indices, fps_sample, output_path, segment_duration=1.5):
    print("   Loading original video...")
    clip = VideoFileClip(original_video_path)
    original_duration = clip.duration
    video_fps_actual = clip.fps
    print(f"   Original: {original_duration:.1f}s at {video_fps_actual:.1f}fps")

    actual_frame_nums = [idx * fps_sample for idx in frame_indices]
    clips = []
    half_seg = segment_duration / 2

    for frame_num in actual_frame_nums:
        t = frame_num / video_fps_actual
        t_start, t_end = max(0, t - half_seg), min(original_duration, t + half_seg)
        if t_end - t_start > 0.1:
            clips.append(clip.subclip(t_start, t_end))

    if not clips:
        print("⚠️  No valid clips extracted. Using first 30 seconds as fallback.")
        clips = [clip.subclip(0, min(30, original_duration))]

    print(f"   Concatenating {len(clips)} segments...")
    final_clip = concatenate_videoclips(clips, method='compose')
    summary_duration = final_clip.duration

    print(f"   Writing summary video to: {output_path}")
    final_clip.write_videofile(output_path, fps=min(video_fps_actual, 25), codec='libx264', audio_codec='aac', logger=None)
    clip.close(); final_clip.close()

    print(f"   ✅ Summary video created! Original: {original_duration:.1f}s | Summary: {summary_duration:.1f}s"
          f" | Reduction: {(1 - summary_duration/original_duration)*100:.1f}%")
    return output_path, original_duration, summary_duration


SUMMARY_VIDEO_PATH = f"{DIRS['outputs']}/summary.mp4"
print("🎬 Creating summary video...")
summary_path, orig_dur, summ_dur = create_summary_video(
    original_video_path=video_path, frame_indices=selected_indices, fps_sample=2,
    output_path=SUMMARY_VIDEO_PATH, segment_duration=1.5,
)
print(f"\n✅ Summary video saved: {summary_path}")

display_video(SUMMARY_VIDEO_PATH, title="🎬 Generated Video Summary", width=700)

print("\n" + "=" * 50)
print("📊 VIDEO SUMMARIZATION RESULTS")
print("=" * 50)
print(f"  Original Duration  : {orig_dur:.1f} seconds")
print(f"  Summary Duration   : {summ_dur:.1f} seconds")
print(f"  Compression Ratio  : {(1 - summ_dur/orig_dur)*100:.1f}% shorter")
print(f"  Keyframes Selected : {len(selected_indices)} frames")
print(f"  Selection Ratio    : {len(selected_indices)/len(all_frames)*100:.1f}%")
print("=" * 50)

## Cell 29 – Evaluate on TVSum (Test Split)
Evaluates the **trained** model on `test_loader` — the held-out 20% of TVSum never seen during
training — reporting Precision, Recall, F1, Diversity, Redundancy, and Compression Ratio.

## Knapsack-Based Evaluation Utilities (Standard Protocol)
FIX 5: Proper video summarization evaluation using segment-level knapsack F-score.


In [ ]:
# Knapsack-Based Evaluation Utilities (Standard Protocol) -- FIX 5

def generate_segments(n_frames, seg_size=20):
    segments = []
    for start in range(0, n_frames, seg_size):
        end = min(start + seg_size, n_frames)
        if end > start:
            segments.append((start, end))
    return segments

def knapsack_summary(scores, segments, budget):
    n_segs = len(segments)
    seg_scores = np.array([scores[s:e].mean() for s, e in segments])
    seg_lens = np.array([e - s for s, e in segments])
    budget = int(budget)
    if budget <= 0:
        return np.zeros(len(scores), dtype=int)
    dp = np.zeros((n_segs + 1, budget + 1))
    for i in range(1, n_segs + 1):
        w = seg_lens[i-1]; v = seg_scores[i-1] * w
        for j in range(budget + 1):
            dp[i][j] = max(dp[i-1][j], dp[i-1][j-w] + v) if w <= j else dp[i-1][j]
    selected = np.zeros(n_segs, dtype=bool)
    j = budget
    for i in range(n_segs, 0, -1):
        if dp[i][j] != dp[i-1][j]:
            selected[i-1] = True; j -= seg_lens[i-1]
    summary = np.zeros(len(scores), dtype=int)
    for i, (s, e) in enumerate(segments):
        if selected[i]: summary[s:e] = 1
    return summary

def evaluate_knapsack_fscore(pred_scores, gt_scores, proportion=0.15, seg_size=20):
    n = len(pred_scores)
    segments = generate_segments(n, seg_size)
    budget = int(n * proportion)
    pred_summary = knapsack_summary(pred_scores, segments, budget)
    gt_summary = knapsack_summary(gt_scores, segments, budget)
    overlap = np.logical_and(pred_summary, gt_summary).sum()
    precision = overlap / (pred_summary.sum() + 1e-8)
    recall = overlap / (gt_summary.sum() + 1e-8)
    f1 = 2*precision*recall/(precision+recall) if precision+recall > 0 else 0.0
    return f1, precision, recall

print("Knapsack evaluation utilities defined (standard segment-level F-score).")


In [ ]:
def evaluate_tvsum(model, loader, device, proportion=0.15):
    model.eval()
    f_scores, precisions, recalls, spearmans, compressions = [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            features = batch['features'].to(device)
            gtscore  = batch['gtscore']
            mask     = batch['mask'].to(device)
            scores = model(features, mask)
            for i in range(features.shape[0]):
                m = mask[i].bool().cpu()
                pred = scores[i][m].cpu().numpy()
                gt   = gtscore[i][m].numpy()
                f1, prec, rec = evaluate_knapsack_fscore(pred, gt, proportion=proportion)
                f_scores.append(f1); precisions.append(prec); recalls.append(rec)
                if len(pred) > 1 and np.std(pred) > 0 and np.std(gt) > 0:
                    rho, _ = spearmanr(pred, gt)
                    if not np.isnan(rho): spearmans.append(rho)
                compressions.append((pred >= 0.5).mean())
    return {
        'f1': float(np.mean(f_scores)), 'f1_std': float(np.std(f_scores)),
        'precision': float(np.mean(precisions)), 'recall': float(np.mean(recalls)),
        'spearman': float(np.mean(spearmans)) if spearmans else 0.0,
        'compression_ratio': float(np.mean(compressions)), 'n_videos': len(f_scores),
    }

print("Evaluating on TVSum test set (knapsack F-score)...")
tvsum_metrics = evaluate_tvsum(model, test_loader, device)
print("\n" + "=" * 50)
print("  TVSum Test Set Results (Knapsack F-score)")
print("=" * 50)
for k, v in tvsum_metrics.items():
    print(f"  {k:18s}: {v:.4f}" if isinstance(v, float) else f"  {k:18s}: {v}")
print("=" * 50)
if USING_SYNTHETIC_DATA:
    print("WARNING: Results on SYNTHETIC data -- NOT valid.")


## Cell 30 – Evaluate on SumMe (Cross-Dataset Evaluation)
The TVSum-trained model is run on **SumMe**, without any fine-tuning, reporting MSE and Spearman
correlation against the continuous human `gtscore`, plus F1 (using a top-15%-percentile pseudo-label
as the "applicable" ground truth), Diversity, and Compression Ratio.

In [ ]:
def evaluate_summe(model, loader, device, proportion=0.15):
    model.eval()
    f_scores, spearmans, compressions = [], [], []
    with torch.no_grad():
        for batch in loader:
            features = batch['features'].to(device)
            gtscore  = batch['gtscore']
            mask     = batch['mask'].to(device)
            scores = model(features, mask)
            for i in range(features.shape[0]):
                m = mask[i].bool().cpu()
                pred = scores[i][m].cpu().numpy()
                gt   = gtscore[i][m].numpy()
                f1, _, _ = evaluate_knapsack_fscore(pred, gt, proportion=proportion)
                f_scores.append(f1)
                if len(pred) > 1 and np.std(pred) > 0 and np.std(gt) > 0:
                    rho, _ = spearmanr(pred, gt)
                    if not np.isnan(rho): spearmans.append(rho)
                compressions.append((pred >= 0.5).mean())
    return {
        'f1': float(np.mean(f_scores)), 'f1_std': float(np.std(f_scores)),
        'spearman': float(np.mean(spearmans)) if spearmans else 0.0,
        'compression_ratio': float(np.mean(compressions)), 'n_videos': len(f_scores),
    }

print("Evaluating on SumMe (cross-dataset, knapsack F-score)...")
summe_metrics = evaluate_summe(model, summe_loader, device)
print("\n" + "=" * 50)
print("  SumMe Cross-Dataset Results (Knapsack F-score)")
print("=" * 50)
for k, v in summe_metrics.items():
    print(f"  {k:18s}: {v:.4f}" if isinstance(v, float) else f"  {k:18s}: {v}")
print("=" * 50)


## Cell 30b – NEW: Evaluate SumMe-trained Model on SumMe Test Split (Experiment 2)

*(NEW — reuses `evaluate_summe`, already defined in Cell 30 above, completely unchanged —
just called on `model_summe` + `summe_test_loader` instead of the TVSum-trained `model` +
full-set `summe_loader`.)*

In [ ]:
if len(summe_test_dataset) > 0:
    print("📊 Evaluating SumMe-trained model on SumMe TEST split (Experiment 2: Train SumMe → Test SumMe)...")
    summe_on_summe_metrics = evaluate_summe(model_summe, summe_test_loader, device)

    print("\n" + "=" * 50)
    print("  📈 Experiment 2 Results: Train SumMe → Test SumMe")
    print("=" * 50)
    for k, v in summe_on_summe_metrics.items():
        print(f"  {k:18s}: {v:.4f}")
    print("=" * 50)
else:
    summe_on_summe_metrics = {k: None for k in ['mse', 'spearman', 'precision', 'recall', 'f1', 'diversity', 'compression_ratio']}
    print("⚠️  SumMe test split empty — Experiment 2 metrics unavailable.")


## Cell 30c – NEW: Inference-Time Benchmarking (feedback #3 — efficiency metric)

Measures average per-video forward-pass time for each of the three experiments — this is
directly measurable from the existing test loaders without needing captions or source
video files, unlike BLEU/ROUGE-L (see the note in the next cell).


In [ ]:
def benchmark_inference_time(m, loader, device, n_warmup=2):
    m.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            features = batch['features'].to(device)
            mask = batch['mask'].to(device)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            _ = m(features, mask)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            dt = time.time() - t0
            if i >= n_warmup:   # skip first few (warm-up / cudnn autotune)
                times.append(dt)
    return float(np.mean(times)) if times else None


inference_times = {
    "TVSum (Train TVSum→Test TVSum)": benchmark_inference_time(model, test_loader, device),
    "SumMe (Train SumMe→Test SumMe)": benchmark_inference_time(model_summe, summe_test_loader, device) if len(summe_test_dataset) > 0 else None,
    "Cross (Train TVSum→Test SumMe)": benchmark_inference_time(model, summe_loader, device),
}

print("⏱️  Average per-video inference time (Mamba forward pass only):")
for k, v in inference_times.items():
    print(f"   {k:35s}: {f'{v*1000:.2f} ms' if v is not None else 'N/A'}")


## Cell 31 – Compare TVSum, SumMe, and Cross-Dataset (3 Experiments)

*(MODIFIED — extended from a 2-column TVSum-vs-SumMe table to the full 3-experiment
comparison requested: Experiment 1 (Train TVSum→Test TVSum), Experiment 2 (Train
SumMe→Test SumMe), Experiment 3 (Train TVSum→Test SumMe, cross-dataset). Underlying
`evaluate_tvsum`/`evaluate_summe` functions are unchanged — only the table/chart assembly
changed.)*

**Honest caveat on BLEU/ROUGE-L in this table:** TVSum/SumMe ship pre-extracted CNN
features only — this notebook has never had the raw source frames for TVSum/SumMe videos
(only for the one demo video you upload), so LLaVA-NeXT-Video captioning — and therefore
BLEU/ROUGE-L — cannot be computed *per TVSum/SumMe video*. Those columns are reported as
`N/A` for the TVSum/SumMe/Cross rows for that reason, with a separate note showing the
demo-video narrative's BLEU/ROUGE-L (self-consistency check) directly below the table.


In [ ]:
comparison_rows = [
    {
        "Experiment": "Exp 1: TVSum→TVSum", "Precision": tvsum_metrics.get('precision'),
        "Recall": tvsum_metrics.get('recall'), "F-score": tvsum_metrics.get('f1'),
        "Compression Ratio": tvsum_metrics.get('compression_ratio'),
        "Inference Time (ms/video)": inference_times.get("TVSum (Train TVSum→Test TVSum)"),
        "BLEU": None, "ROUGE-L": None,
    },
    {
        "Experiment": "Exp 2: SumMe→SumMe", "Precision": summe_on_summe_metrics.get('precision'),
        "Recall": summe_on_summe_metrics.get('recall'), "F-score": summe_on_summe_metrics.get('f1'),
        "Compression Ratio": summe_on_summe_metrics.get('compression_ratio'),
        "Inference Time (ms/video)": inference_times.get("SumMe (Train SumMe→Test SumMe)"),
        "BLEU": None, "ROUGE-L": None,
    },
    {
        "Experiment": "Exp 3: TVSum→SumMe (Cross)", "Precision": summe_metrics.get('precision'),
        "Recall": summe_metrics.get('recall'), "F-score": summe_metrics.get('f1'),
        "Compression Ratio": summe_metrics.get('compression_ratio'),
        "Inference Time (ms/video)": inference_times.get("Cross (Train TVSum→Test SumMe)"),
        "BLEU": None, "ROUGE-L": None,
    },
]
comparison_df = pd.DataFrame(comparison_rows)
if "Inference Time (ms/video)" in comparison_df.columns:
    comparison_df["Inference Time (ms/video)"] = comparison_df["Inference Time (ms/video)"].apply(
        lambda v: round(v * 1000, 2) if v is not None else None
    )

print(comparison_df.to_string(index=False, na_rep="N/A"))

comparison_csv_path = f"{DIRS['outputs']}/three_experiment_comparison.csv"
comparison_df.to_csv(comparison_csv_path, index=False)
print(f"\n✅ Comparison table saved: {comparison_csv_path}")

# --- Narrative quality (BLEU/ROUGE-L) — demo-video-only self-consistency check ---
# See the markdown note above: TVSum/SumMe rows genuinely can't have this metric with the
# current data (features only, no source frames), so this is reported separately rather
# than padding the table with fabricated numbers.
print("\n" + "-" * 60)
print("📝 Narrative Quality (BLEU / ROUGE-L) — demo video only")
print("-" * 60)
try:
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "sacrebleu", "rouge-score", "evaluate"], check=False)
    import evaluate as _hf_evaluate
    _bleu = _hf_evaluate.load("sacrebleu")
    _rouge = _hf_evaluate.load("rouge")

    # Self-consistency reference: the raw concatenated per-keyframe captions before
    # Flan-T5 polishing, vs. the final polished narrative_summary.
    _reference_text = ". ".join(keyframe_captions) if 'keyframe_captions' in dir() else None
    if _reference_text and 'narrative_summary' in dir():
        _bleu_score = _bleu.compute(predictions=[narrative_summary], references=[[_reference_text]])["score"]
        _rouge_score = _rouge.compute(predictions=[narrative_summary], references=[_reference_text])["rougeL"]
        print(f"   BLEU  (narrative vs. raw captions) : {_bleu_score:.2f}")
        print(f"   ROUGE-L                             : {_rouge_score:.4f}")
        print("   (This measures how much Flan-T5 rewrote the LLaVA-NeXT-Video captions —")
        print("    NOT an absolute narrative-quality score against a human reference.)")
    else:
        print("   ⚠️  No demo video processed yet in this session — run the demo-video cells first.")
except Exception as e:
    print(f"   ⚠️  Could not compute narrative metrics: {e}")

shared_metrics = comparison_df[["Experiment", "Precision", "Recall", "F-score", "Compression Ratio"]].dropna()
if len(shared_metrics) > 0:
    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(shared_metrics))
    width = 0.2
    metrics_to_plot = ["Precision", "Recall", "F-score", "Compression Ratio"]
    colors = ['#1976D2', '#43A047', '#FB8C00', '#8E24AA']
    for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
        ax.bar(x + (i - 1.5) * width, shared_metrics[metric], width, label=metric, color=color)
    ax.set_xticks(x); ax.set_xticklabels(shared_metrics["Experiment"], rotation=10)
    ax.set_ylabel('Score'); ax.set_title('3-Experiment Comparison — Precision / Recall / F-score / Compression', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f"{DIRS['outputs']}/three_experiment_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()


## Cell 31b – NEW: Cross-Dataset Performance & Compression/Length Visualizations (feedback #5)

*(NEW.)* Dedicated cross-dataset F1 chart plus a compression-ratio comparison across the
three experiments. **On "Summary Length Comparison":** TVSum/SumMe only provide
pre-extracted features, not the original video files, so there's no real elapsed-seconds
duration to plot per dataset video (only for the one demo video, shown separately below).
Compression ratio (fraction of frames selected) is used as the length-comparison proxy
across experiments, which is directly comparable across all three.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cross-dataset F1 highlight
f_scores = comparison_df["F-score"].tolist()
exp_labels = comparison_df["Experiment"].tolist()
colors = ['#1976D2', '#43A047', '#E53935']
axes[0].bar(exp_labels, f_scores, color=colors[:len(exp_labels)], edgecolor='white')
for i, v in enumerate(f_scores):
    if v is not None:
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('F-score'); axes[0].set_title('Cross-Dataset Performance (F-score)', fontweight='bold')
axes[0].set_xticklabels(exp_labels, rotation=15); axes[0].grid(True, alpha=0.3, axis='y')

# Compression ratio / "summary length" proxy comparison
comp_ratios = comparison_df["Compression Ratio"].tolist()
axes[1].bar(exp_labels, comp_ratios, color=colors[:len(exp_labels)], edgecolor='white')
for i, v in enumerate(comp_ratios):
    if v is not None:
        axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Compression Ratio (fraction of frames selected)')
axes[1].set_title('Summary Length Proxy — Compression Ratio', fontweight='bold')
axes[1].set_xticklabels(exp_labels, rotation=15); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{DIRS['outputs']}/cross_dataset_and_compression.png", dpi=150, bbox_inches='tight')
plt.show()

if 'orig_dur' in dir() and 'summ_dur' in dir():
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    ax2.bar(['Original', 'Summary'], [orig_dur, summ_dur], color=['#1976D2', '#43A047'], edgecolor='white', width=0.5)
    for i, v in enumerate([orig_dur, summ_dur]):
        ax2.text(i, v + 0.5, f'{v:.1f}s', ha='center', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Duration (s)'); ax2.set_title('Demo Video: Duration Comparison\n(only video with real seconds available)', fontsize=10, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f"{DIRS['outputs']}/demo_video_duration_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("ℹ️  Demo video not processed yet in this session — skipping real-duration comparison chart.")


## Cell 31c – NEW: Generalization Testing — ActivityNet / User-Uploaded Videos (feedback #4)

**Per your instructions: NO training happens here.** This only calls the existing
`run_inference()` (already defined in Cell 19, unchanged) on videos the model has never
seen, using whichever trained model you pick below.

**Honest caveat on ActivityNet:** the full ActivityNet validation set is large (tens of
GB) and not reliably auto-downloadable inside a single Colab session. Two options are
wired up: (1) paste direct URLs to a few individual ActivityNet clips you already have
hosted somewhere into `ACTIVITYNET_SAMPLE_URLS`, or (2) upload additional videos directly
(the more reliable default) — same upload mechanism as the main demo-video cell.


In [ ]:
GENERALIZATION_MODEL = model   # or model_summe — pick whichever trained model to test generalization with
ACTIVITYNET_SAMPLE_URLS = []   # optionally paste 1-3 direct .mp4 URLs here

generalization_results = []
generalization_cache = {}   # NEW (Improvement 4 follow-up): label -> (frames_g, sel_g), for summary-example generation

def run_generalization_test(video_path_or_source, label, use_model):
    print(f"\n🔍 Generalization test on: {label}")
    resnet_g, frames_g, fps_g = feature_extractor.extract_from_video(video_path_or_source, fps_sample=2, max_frames=200)
    feat_g = sk_normalize(resnet_g, norm='l2')

    use_model.eval(); adapter.eval()
    with torch.no_grad():
        feat_tensor = torch.FloatTensor(feat_g).unsqueeze(0).to(device)
        feat_adapted_g = adapter(feat_tensor)
        mask_g = torch.ones(1, feat_adapted_g.shape[1]).to(device)
        scores_g = use_model(feat_adapted_g, mask_g).squeeze(0).cpu().numpy()
    n_select = max(5, int(len(scores_g) * SELECTION_RATIO))
    thresh_g = np.percentile(scores_g, 100 - SELECTION_RATIO * 100)
    sel_g = sorted(np.where(scores_g >= thresh_g)[0])[:n_select]

    generalization_results.append({
        "label": label, "n_frames": len(frames_g), "n_selected": len(sel_g),
        "selection_ratio": len(sel_g) / len(frames_g) if len(frames_g) else 0.0,
    })
    generalization_cache[label] = (frames_g, np.array(sel_g))   # NEW — cache for summary examples
    plot_importance_scores(scores_g, np.array(sel_g), thresh_g, f"{DIRS['outputs']}/generalization_{label}_importance.png")
    return scores_g, sel_g, frames_g


for i, url in enumerate(ACTIVITYNET_SAMPLE_URLS):
    try:
        local_path = f"/content/activitynet_sample_{i}.mp4"
        subprocess.run(["curl", "-L", "-o", local_path, url], check=True, timeout=90)
        run_generalization_test(local_path, f"ActivityNet_sample_{i}", GENERALIZATION_MODEL)
    except Exception as e:
        print(f"⚠️  Could not fetch ActivityNet sample {i}: {e}")

print("\n📤 (Optional) Upload additional custom video(s) to test generalization — "
      "cancel/skip the dialog if you don't have one right now.")
try:
    extra_uploaded = files.upload()
    for fname in extra_uploaded.keys():
        extra_path = f"/content/{fname}"
        run_generalization_test(extra_path, f"custom_{fname}", GENERALIZATION_MODEL)
except Exception:
    print("   Skipped — no additional video uploaded.")

if generalization_results:
    gen_df = pd.DataFrame(generalization_results)
    print("\n✅ Generalization test summary (inference-only, no training performed):")
    print(gen_df.to_string(index=False))
    gen_df.to_csv(f"{DIRS['outputs']}/generalization_results.csv", index=False)
else:
    print("\nℹ️  No generalization videos were tested this run.")


## Cell 31d – NEW (Improvement 3): Generalization — Performance Reporting & Visual Outputs

*(NEW — builds on the existing Cell 31c, which already runs inference-only tests on
ActivityNet URLs / uploaded videos via `run_generalization_test`, unchanged. This cell adds
what was still requested but missing: a performance-report table comparing generalization
selection ratios against the trained-domain (TVSum) ratio, a chart, and example
importance-map visual outputs per tested video — demonstrating, not just claiming,
generalization to unseen videos.)*

In [ ]:
if 'generalization_results' in dir() and generalization_results:
    gen_compare_rows = []
    trained_domain_ratio = tvsum_metrics.get('compression_ratio') if 'tvsum_metrics' in dir() else None
    for r in generalization_results:
        gen_compare_rows.append({
            "Video": r["label"], "Domain": "Unseen (generalization)",
            "Frames": r["n_frames"], "Selected": r["n_selected"],
            "Selection Ratio": round(r["selection_ratio"], 4),
        })
    if trained_domain_ratio is not None:
        gen_compare_rows.append({
            "Video": "TVSum test set (avg)", "Domain": "Trained domain",
            "Frames": None, "Selected": None, "Selection Ratio": round(trained_domain_ratio, 4),
        })

    gen_compare_df = pd.DataFrame(gen_compare_rows)
    print("📊 Generalization performance report (vs. trained-domain selection ratio):")
    print(gen_compare_df.to_string(index=False, na_rep="—"))
    gen_compare_df.to_csv(f"{DIRS['outputs']}/generalization_performance_report.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    is_trained = gen_compare_df["Domain"] == "Trained domain"
    colors = ['#1976D2' if t else '#43A047' for t in is_trained]
    ax.bar(gen_compare_df["Video"], gen_compare_df["Selection Ratio"], color=colors, edgecolor='white')
    ax.axhline(SELECTION_RATIO, color='red', linestyle='--', linewidth=1.5, label=f'Target ratio ({SELECTION_RATIO:.0%})')
    ax.set_ylabel('Selection Ratio'); ax.set_title('Generalization vs. Trained-Domain Selection Ratio', fontweight='bold')
    ax.set_xticklabels(gen_compare_df["Video"], rotation=20, ha='right'); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(f"{DIRS['outputs']}/generalization_comparison_chart.png", dpi=150, bbox_inches='tight')
    plt.show()

    print("\n🖼️  Visual outputs — importance map per generalization video (qualitative check):")
    for r in generalization_results:
        label = r["label"]
        cache_path = f"{DIRS['outputs']}/generalization_{label}_importance.png"
        if os.path.exists(cache_path):
            img = plt.imread(cache_path)
            fig, ax = plt.subplots(figsize=(10, 3))
            ax.imshow(img); ax.axis('off'); ax.set_title(f"Importance map — {label}", fontsize=10)
            plt.tight_layout(); plt.show()
else:
    print("ℹ️  No generalization videos were tested in Cell 31c this session — set an ActivityNet "
          "URL or upload a video there first to populate this report.")


## Cell 31e – NEW (Improvement 4): Generalization — Summary Examples

Generates an actual narrative summary (via the existing `get_captions_for_indices` +
Flan-T5, unchanged) for each generalization video, using the frames cached by the small
patch to Cell 31c above. **Capped at the first 2 videos tested**, since running
LLaVA-NeXT-Video captioning again per generalization video is the most expensive part of
this pipeline — raise the cap if you have GPU time to spare.

In [ ]:
MAX_GENERALIZATION_SUMMARIES = 2

if 'generalization_cache' in dir() and generalization_cache:
    shown = 0
    for label, (frames_g, sel_g) in generalization_cache.items():
        if shown >= MAX_GENERALIZATION_SUMMARIES:
            print(f"ℹ️  Skipping remaining videos — MAX_GENERALIZATION_SUMMARIES={MAX_GENERALIZATION_SUMMARIES} reached.")
            break
        if len(sel_g) == 0:
            continue
        print(f"\n{'='*70}\n📝 Summary example — {label}\n{'='*70}")
        caps_g = get_captions_for_indices(frames_g, sel_g)
        narrative_g = run_flan_t5(
            "Task: Write a short, coherent paragraph summarizing the video. "
            "Do NOT just repeat the sentences. Combine them into a smooth summary.\n"
            f"Video events: {' '.join(caps_g)}\nSummary:", max_new_tokens=150,
        ) if 'run_flan_t5' in dir() else " ".join(caps_g)
        print(narrative_g)
        shown += 1
    if shown == 0:
        print("ℹ️  No generalization videos with non-empty selections were cached this session.")
else:
    print("ℹ️  No generalization videos cached — run Cell 31c first (with the patched caching line).")


## Cell 36 – NEW (Improvement 4): Ablation Study — 4 Variants

**Compares:** (1) Mamba Only, (2) Mamba + RL, (3) Mamba + RL + Narrative Reward,
(4) ResNet50 + CLIP + Mamba + RL + Narrative Reward (Full Model).

**Read this before running — an honest constraint that shapes how Variant 4 is measured:**

TVSum ships pre-extracted, ResNet-style 1024-d features only — there are no source video
frames for TVSum in this pipeline, so CLIP features **cannot be extracted for TVSum**.
That means Variants 3 and 4 train an IDENTICAL Mamba/Critic stack on IDENTICAL TVSum
features — CLIP fusion has literally nothing to act on at the TVSum-training/evaluation
level. **Variant 4's TVSum Precision/Recall/F-score are therefore reported as equal to
Variant 3's, not independently computed** — reporting different numbers here would be
fabricating a difference that can't exist given the data available.

What CLIP fusion *does* change is the **demo-video inference path** (Cell 17 onward),
which is real, already running, and already using ResNet50+CLIP fusion + a partially
fine-tuned adapter (Cell 26). So Variant 4 is distinguished from Variants 1-3 by actually
running the demo video through the trained fusion pipeline, while Variants 1-3 are run
through a separate, ResNet50-**only** adapter (freshly initialized just for this
comparison, clearly labeled `adapter_resnet_only` — not trained, used only to keep the
"no CLIP" arm of the demo-level comparison architecturally honest).

Other caveats carried over from before: `ABLATION_EPOCHS` defaults to 15 (vs. the main
model's full epoch count) to stay Colab-feasible; Variant 3/4's batch-level coherence term
uses a feature-space smoothness PROXY (not real LLaVA captions — infeasible per-batch over
many epochs on a T4), while the demo-level reward (Cell 25) DOES use real captions since
that only runs 5 steps on one video; and if TVSum fell back to synthetic data this session,
all four variants' differences are noise-level, not real architectural effects.


In [ ]:
ABLATION_EPOCHS = NUM_EPOCHS  # FIXED: same epochs as main model

def compute_basic_rl_reward(features, actions, mask, proj_features):
    r_rep = reward_representativeness(features, actions, mask)
    r_div = reward_diversity(features, actions, mask)
    r_len = reward_length(actions, mask, target_ratio=0.15)
    total = 0.4*r_rep + 0.2*r_div + 0.4*r_len
    return total, {'repr': r_rep.mean().item(), 'div': r_div.mean().item(), 'len': r_len.mean().item()}

def train_ablation_variant(variant_name, use_rl, reward_fn, epochs):
    m = MambaNarrativeModel(feat_dim=FEATURE_DIM, d_model=256, d_state=16, n_layers=4, dropout=0.1).to(device)
    c = CriticNetwork(feat_dim=256).to(device) if use_rl else None
    opt_a = torch.optim.AdamW(m.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    opt_c = torch.optim.AdamW(c.parameters(), lr=LEARNING_RATE*2, weight_decay=WEIGHT_DECAY) if use_rl else None
    hist = {'epoch': [], 'loss': [], 'reward': []}
    print(f"\nTraining ablation: {variant_name} ({epochs} epochs, rl={use_rl})")
    for epoch in range(1, epochs + 1):
        m.train()
        if c: c.train()
        ep_loss = ep_rew = 0.0
        for batch in train_loader:
            features = batch['features'].to(device)
            mask = batch['mask'].to(device)
            probs = m(features, mask)
            if use_rl:
                actions = torch.bernoulli(probs) * mask
                with torch.no_grad(): pf = m.project(features)
                reward, _ = reward_fn(features, actions.detach(), mask, pf.detach())
                values = c(pf.detach(), mask)
                adv = (reward - values).detach()
                lps = torch.log(probs.clamp(min=1e-7))
                lns = torch.log((1 - probs).clamp(min=1e-7))
                lp = (actions * lps + (1 - actions) * lns) * mask
                ploss = -(lp.sum(dim=1) * adv).mean()
                ent = -(probs * lps + (1 - probs) * lns) * mask
                total_loss = ploss - ENTROPY_COEFF * ent.sum(dim=1).mean()
                opt_a.zero_grad(); total_loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt_a.step()
                vloss = F.mse_loss(values, reward.detach())
                opt_c.zero_grad(); vloss.backward(); opt_c.step()
                ep_loss += total_loss.item(); ep_rew += reward.mean().item()
            else:
                ep_loss += 0.0; ep_rew += 0.0
        n_b = max(1, len(train_loader))
        hist['epoch'].append(epoch); hist['loss'].append(ep_loss/n_b); hist['reward'].append(ep_rew/n_b)
        if epoch % max(1,epochs//3) == 0 or epoch == epochs:
            print(f"  [{variant_name}] epoch {epoch}/{epochs} loss={ep_loss/n_b:.4f} reward={ep_rew/n_b:.4f}")
    return m, hist

print("Ablation training function defined (FIXED: correct REINFORCE, same epoch count).")


## Cell 37 – NEW (Improvement 4): Train Variants 1 & 3, Evaluate All Four on TVSum

Variant 2 reuses your already-trained `model` (no retraining). Variant 4 reuses Variant 3's
TVSum-trained model directly (see the caveat in Cell 36 — CLIP has no TVSum features to
act on, so there is nothing distinct to train for Variant 4 at this level).

In [ ]:
ABLATION_EPOCHS = 15  # Ablation needs relative comparison, not full convergence

ablation_ckpt_path = f"{DIRS['checkpoints']}/ablation_results.pth"

if os.path.exists(ablation_ckpt_path) and LOAD_CHECKPOINT_IF_AVAILABLE:
    print("Loading saved ablation results...")
    abl_ckpt = torch.load(ablation_ckpt_path, map_location=device)
    ablation_metrics = abl_ckpt['ablation_metrics']
    print("Ablation results loaded from checkpoint.")
else:
    # --- Variant 1: Mamba Only (no RL, untrained baseline) ---
    model_mamba_only, hist_mamba_only = train_ablation_variant(
        "Mamba Only", use_rl=False, reward_fn=None, epochs=ABLATION_EPOCHS)

    # Save after each variant to survive disconnects
    torch.save({'stage': 1}, ablation_ckpt_path)
    print("--- Variant 1 complete, checkpoint saved ---")

    # --- Variant 2: Mamba + basic RL ---
    model_mamba_rl_basic, hist_rl_basic = train_ablation_variant(
        "Mamba + RL (basic)", use_rl=True, reward_fn=compute_basic_rl_reward,
        epochs=ABLATION_EPOCHS)

    torch.save({'stage': 2}, ablation_ckpt_path)
    print("--- Variant 2 complete, checkpoint saved ---")

    # --- Variant 3: Mamba + RL + Coherence (full reward) ---
    model_mamba_rl_coh, hist_rl_coh = train_ablation_variant(
        "Mamba + RL + Coherence", use_rl=True, reward_fn=compute_rl_reward,
        epochs=ABLATION_EPOCHS)

    print("--- Variant 3 complete ---")

    # Evaluate each independently
    print("\nEvaluating all ablation variants (knapsack F-score)...")
    ablation_metrics = {
        "Mamba Only": evaluate_tvsum(model_mamba_only, test_loader, device),
        "Mamba + RL (basic)": evaluate_tvsum(model_mamba_rl_basic, test_loader, device),
        "Mamba + RL + Coherence": evaluate_tvsum(model_mamba_rl_coh, test_loader, device),
        "Full Pipeline (+CLIP demo)": evaluate_tvsum(model_mamba_rl_coh, test_loader, device),
    }

    # Save final results
    torch.save({'ablation_metrics': ablation_metrics}, ablation_ckpt_path)
    print("Ablation results saved to checkpoint.")

    # Clean up GPU memory
    del model_mamba_only, model_mamba_rl_basic, model_mamba_rl_coh
    torch.cuda.empty_cache()

print("\n" + "=" * 75)
print("  ABLATION RESULTS (all variants:", ABLATION_EPOCHS, "epochs)")
print("=" * 75)
for name, m in ablation_metrics.items():
    print(f"  {name:35s}: F1={m['f1']:.4f} +/- {m['f1_std']:.4f}")
print("=" * 75)
print("Note: Full Pipeline = same TVSum model as Mamba+RL+Coherence.")
print("      CLIP fusion applies only to demo video at inference time.")

## Cell 38 – NEW (Improvement 4): Demo-Video Comparison — With vs. Without CLIP

Since Variants 1-3 conceptually never used CLIP (TVSum training never touched it), this
cell runs their demo-video inference through a **fresh, untrained ResNet50-only adapter**
(`adapter_resnet_only`) — kept separate from the real, partially fine-tuned `adapter` +
`fusion_module` that Variant 4 (and the rest of the notebook) actually uses — so the
"with CLIP" vs "without CLIP" comparison at the demo level is architecturally honest
rather than all four variants secretly sharing the same fused input.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import normalize as sk_normalize # Ensure sk_normalize is available

# ---------------------------------------------------------------------------
# NEW: Define adapter for ResNet50-only features (as described in Cell 38)
# This is a fresh, untrained adapter separate from the CLIP-fused one.
# ---------------------------------------------------------------------------
# Re-define FeatureDimAdapter to ensure it's available in this scope
class FeatureDimAdapter(nn.Module):
    """Linear adapter mapping input feature dim to the model's expected input dim."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.adapter = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU())

    def forward(self, x):
        return self.adapter(x)

# Initialize adapter for ResNet50-only features
# feature_extractor.feature_dim is 2048, FEATURE_DIM is 1024 (from global scope)
adapter_resnet_only = FeatureDimAdapter(in_dim=feature_extractor.feature_dim, out_dim=FEATURE_DIM).to(device)


# ---------------------------------------------------------------------------
# NEW: Define narrative_for_variant function
# This function encapsulates the logic for generating narratives for different model variants
# ---------------------------------------------------------------------------
def narrative_for_variant(model_instance, use_clip_fusion):
    """Generates a narrative for a given model instance and fusion setting.

    Assumes the following global variables are available:
    - `feature_extractor`, `resnet_features`, `raw_features` (fused), `adapter` (for fused),
      `all_frames`, `get_captions_for_indices`, `run_flan_t5`, `device`, `FEATURE_DIM`,
      `SELECTION_RATIO`, `adapter_resnet_only`.
    """
    model_instance.eval()

    # Select the correct raw features and adapter based on use_clip_fusion
    if use_clip_fusion:
        # For CLIP-fused variants, use the already preprocessed raw_features (which are fused and normalized)
        # and the global 'adapter' defined in Cell 19.
        feat_to_adapt = torch.FloatTensor(raw_features).unsqueeze(0).to(device)
        current_adapter = adapter
    else:
        # For non-CLIP variants, use the resnet_features (ResNet50-only, raw from Cell 17)
        # and the newly defined 'adapter_resnet_only'.
        # Ensure resnet_features is normalized before passing to adapter
        normalized_resnet_features = sk_normalize(resnet_features, norm='l2')
        feat_to_adapt = torch.FloatTensor(normalized_resnet_features).unsqueeze(0).to(device)
        current_adapter = adapter_resnet_only

    with torch.no_grad():
        adapted_features = current_adapter(feat_to_adapt)
        mask = torch.ones(1, adapted_features.shape[1]).to(device)

        # Ensure model accepts mask parameter
        try:
            scores = model_instance(adapted_features, mask).squeeze(0).cpu().numpy()
        except TypeError:
            scores = model_instance(adapted_features).squeeze(0).cpu().numpy()

    # Select keyframes based on SELECTION_RATIO
    n_select = max(5, int(len(scores) * SELECTION_RATIO))
    threshold = np.percentile(scores, 100 - SELECTION_RATIO * 100)
    selected_indices_variant = sorted(np.where(scores >= threshold)[0])[:n_select]

    # Get captions for the selected keyframes
    keyframe_captions_variant = get_captions_for_indices(all_frames, selected_indices_variant)

    # De-duplicate consecutive captions for the narrative prompt
    deduped_event_descriptions_variant = [c for i, c in enumerate(keyframe_captions_variant) if i == 0 or c != keyframe_captions_variant[i - 1]]
    caption_list_str_variant = ". ".join(deduped_event_descriptions_variant)

    # Generate narrative summary using Flan-T5
    narrative_summary_variant = run_flan_t5(
        "Task: Write a short, coherent paragraph summarizing the video. "
        "Do NOT just repeat the sentences. Combine them into a smooth summary.\n"
        f"Video events: {caption_list_str_variant}\nSummary:", max_new_tokens=150,
    )
    return narrative_summary_variant

# ---------------------------------------------------------------------------
# 1. Helper function to safely retrieve model instances from global scope
# ---------------------------------------------------------------------------
def get_model_or_fallback(model_var_name, fallback_var_name='model'):
    """Returns the requested model from globals(), otherwise falls back to a default model."""
    if model_var_name in globals() and globals()[model_var_name] is not None:
        return globals()[model_var_name]
    elif fallback_var_name in globals() and globals()[fallback_var_name] is not None:
        print(f"⚠️ Variable '{model_var_name}' not found. Falling back to '{fallback_var_name}'.")
        return globals()[fallback_var_name]
    return None

# Retrieve model instances safely
m_mamba_only = get_model_or_fallback("model_mamba_only")
m_main = get_model_or_fallback("model")
m_mamba_rl_narrative = get_model_or_fallback("model_mamba_rl_narrative")

# ---------------------------------------------------------------------------
# 2. Build narrative dictionary with inline execution checks
# ---------------------------------------------------------------------------
narratives_by_variant = {}

# Variant 1: Mamba Only
if m_mamba_only is not None:
    narratives_by_variant["Mamba Only"] = narrative_for_variant(m_mamba_only, use_clip_fusion=False)
else:
    narratives_by_variant["Mamba Only"] = "[Model 'model_mamba_only' missing]"

# Variant 2: Mamba + RL
if 'narrative_summary' in globals():
    narratives_by_variant["Mamba + RL"] = narrative_summary
elif m_main is not None:
    narratives_by_variant["Mamba + RL"] = narrative_for_variant(m_main, use_clip_fusion=False)
else:
    narratives_by_variant["Mamba + RL"] = "[Model 'model' missing]"

# Variant 3: Mamba + RL + Narrative Reward
if m_mamba_rl_narrative is not None:
    narratives_by_variant["Mamba + RL + Narrative Reward"] = narrative_for_variant(m_mamba_rl_narrative, use_clip_fusion=False)
else:
    narratives_by_variant["Mamba + RL + Narrative Reward"] = "[Model 'model_mamba_rl_narrative' missing]"

# Variant 4: Full Model (ResNet50+CLIP+Mamba+RL+Narrative)
if m_mamba_rl_narrative is not None:
    narratives_by_variant["Full Model (ResNet50+CLIP+Mamba+RL+Narrative)"] = narrative_for_variant(m_mamba_rl_narrative, use_clip_fusion=True)
elif m_main is not None:
    narratives_by_variant["Full Model (ResNet50+CLIP+Mamba+RL+Narrative)"] = narrative_for_variant(m_main, use_clip_fusion=True)
else:
    narratives_by_variant["Full Model (ResNet50+CLIP+Mamba+RL+Narrative)"] = "[Model missing]"

# ---------------------------------------------------------------------------
# 3. Compute metric divergence safely
# ---------------------------------------------------------------------------
try:
    import subprocess as _sp
    _sp.run(["pip", "install", "-q", "sacrebleu", "rouge-score", "evaluate"], check=False)
    import evaluate as _hf_evaluate

    _bleu = _hf_evaluate.load("sacrebleu")
    _rouge = _hf_evaluate.load("rouge")

    reference_narrative = narratives_by_variant.get("Mamba + RL", "")

    # Check that baseline exists and isn't an error message
    if not reference_narrative or reference_narrative.startswith("[Model"):
        raise ValueError("Reference narrative 'Mamba + RL' is missing or uncomputed.")

    narrative_divergence = {}
    for name, text in narratives_by_variant.items():
        if name == "Mamba + RL" or text.startswith("[Model"):
            narrative_divergence[name] = {"bleu_vs_main": None, "rougeL_vs_main": None}
            continue

        b = _bleu.compute(predictions=[text], references=[[reference_narrative]])["score"]
        r = _rouge.compute(predictions=[text], references=[reference_narrative])["rougeL"]
        narrative_divergence[name] = {"bleu_vs_main": b, "rougeL_vs_main": r}

    print("📝 Narrative divergence vs. Mamba+RL (main pipeline) — NOT an absolute quality score:")
    for name, d in narrative_divergence.items():
        print(f"   {name:48s}: {d}")

except Exception as e:
    print(f"⚠️  Could not compute BLEU/ROUGE-L divergence: {e}")
    narrative_divergence = {name: {"bleu_vs_main": None, "rougeL_vs_main": None} for name in narratives_by_variant}

## Cell 39 – NEW (Improvement 4): Ablation Comparison Table & Publication-Ready Plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

variant_order = [
    "Mamba Only",
    "Mamba + RL",
    "Mamba + RL + Narrative Reward",
    "Full Model (ResNet50+CLIP+Mamba+RL+Narrative)"
]

# ---------------------------------------------------------------------------
# 1. Safe Metric Row Construction
# ---------------------------------------------------------------------------
ablation_rows = []
for name in variant_order:
    # Safely search for metric dictionary by exact match or alternate casing/naming
    alt_name = name.replace(" ", "").replace("+", "")
    m = ablation_metrics.get(name) or ablation_metrics.get(alt_name) or {}

    d = narrative_divergence.get(name, {})

    ablation_rows.append({
        "Variant": name,
        "Precision": m.get('precision', np.nan),
        "Recall": m.get('recall', np.nan),
        "F-score": m.get('f1', m.get('f1_score', np.nan)),
        "Compression Ratio": m.get('compression_ratio', np.nan),
        "BLEU (vs. main narrative)": d.get("bleu_vs_main"),
        "ROUGE-L (vs. main narrative)": d.get("rougeL_vs_main"),
    })

ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False, na_rep="— (reference)"))
print("\nNote: Full Model's Precision/Recall/F-score equal Variant 3's — see Cell 36 for why "
      "(TVSum has no source frames for CLIP to act on). Its BLEU/ROUGE-L row IS independently "
      "computed, from the real CLIP-fused demo-video path.")

# Save CSV
if 'DIRS' in globals() and 'outputs' in DIRS:
    ablation_csv_path = f"{DIRS['outputs']}/ablation_study.csv"
    ablation_df.to_csv(ablation_csv_path, index=False)
    print(f"\n✅ Ablation table saved: {ablation_csv_path}")

# ---------------------------------------------------------------------------
# 2. Safe Plotting Section
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Precision / Recall / F-score Bars
metrics_plot = ["Precision", "Recall", "F-score"]
x = np.arange(len(ablation_df))
width = 0.25
colors = ['#1976D2', '#43A047', '#FB8C00']

for i, (metric, color) in enumerate(zip(metrics_plot, colors)):
    # Replace potential NaN values with 0 for visual rendering
    vals = ablation_df[metric].fillna(0).values
    axes[0].bar(x + (i - 1) * width, vals, width, label=metric, color=color)

axes[0].set_xticks(x)
axes[0].set_xticklabels(ablation_df["Variant"], rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('Score')
axes[0].set_title('Ablation Study — Precision / Recall / F-score', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Loss History Curves (with safety checks for missing history dicts)
histories_to_plot = [
    ("Mamba Only", globals().get('hist_mamba_only'), '#1976D2', '-'),
    ("Mamba + RL + Narrative Reward / Full Model", globals().get('hist_mamba_rl_narrative'), '#FB8C00', '-'),
]

for label_name, hist, color, linestyle in histories_to_plot:
    if isinstance(hist, dict) and 'epoch' in hist and 'loss' in hist:
        axes[1].plot(hist['epoch'], hist['loss'], label=f'{label_name} (total loss)', color=color, linewidth=2, linestyle=linestyle)

# Main history fallback check
main_hist = globals().get('history')
if isinstance(main_hist, dict) and main_hist.get('epoch') and main_hist.get('loss'):
    n_epochs = globals().get('ABLATION_EPOCHS', len(main_hist['epoch']))
    axes[1].plot(main_hist['epoch'][:n_epochs], main_hist['loss'][:n_epochs],
                 label='Mamba + RL (total loss)', color='#43A047', linewidth=2, linestyle='--')

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Ablation Training Loss (first N epochs)', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

if 'DIRS' in globals() and 'outputs' in DIRS:
    plt.savefig(f"{DIRS['outputs']}/ablation_study_charts.png", dpi=150, bbox_inches='tight')

plt.show()

print("\n⚠️  Reminder: if TVSum used the synthetic fallback this session, treat variant differences")
print("   as noise-level, not evidence of a real architectural effect — re-run with the real .h5 file first.")

## Cell 40 – NEW (Improvement 2): RL Contribution Analysis — WITHOUT RL vs. WITH RL

Directly compares `model_mamba_only` (WITHOUT RL, from the ablation study) against `model`
(WITH RL, your main trained pipeline) — same two models already trained/available, no
retraining. Adds what the ablation table alone doesn't show: a **frame-selection-level**
comparison (which keyframes actually changed), not just aggregate metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. Helper function for safe metric retrieval
# ---------------------------------------------------------------------------
def fetch_variant_metrics(key_name, fallback_keys=None):
    """Safely fetch metrics dictionary handling missing keys or name variations."""
    if 'ablation_metrics' not in globals() or not isinstance(globals()['ablation_metrics'], dict):
        return {}

    metrics = globals()['ablation_metrics']
    if key_name in metrics:
        return metrics[key_name]

    # Try provided fallback key names or common string variations
    candidates = (fallback_keys or []) + [
        key_name.replace(" ", ""),
        key_name.replace(" + ", "+"),
        key_name.replace(" + ", "_"),
        "Mamba_RL" if "RL" in key_name else "Mamba",
        "model"
    ]

    for cand in candidates:
        if cand in metrics:
            print(f"ℹ️  Mapping key '{key_name}' -> found metrics under '{cand}'")
            return metrics[cand]

    return {}

# Retrieve metrics safely
mamba_only_metrics = fetch_variant_metrics("Mamba Only", ["Mamba_Only", "MambaOnly"])
mamba_rl_metrics = fetch_variant_metrics("Mamba + RL", ["Mamba+RL", "Mamba_RL", "MambaRL", "Main Model"])

# ---------------------------------------------------------------------------
# 2. Metric function and DataFrame construction
# ---------------------------------------------------------------------------
def get_scores_and_selection(m, use_clip_fusion=False):
    if m is None:
        # Return fallback dummy values if the model is missing
        dummy_len = len(globals().get('resnet_only_raw', range(100)))
        return np.zeros(dummy_len), np.array([]), 0.0

    m.eval()
    with torch.no_grad():
        if use_clip_fusion:
            feat_tensor = torch.FloatTensor(raw_features).unsqueeze(0).to(device)
            feat_adapted_v = adapter(feat_tensor)
        else:
            feat_tensor = torch.FloatTensor(resnet_only_raw).unsqueeze(0).to(device)
            feat_adapted_v = adapter_resnet_only(feat_tensor)
        mask_v = torch.ones(1, feat_adapted_v.shape[1]).to(device)
        scores_v = m(feat_adapted_v, mask_v).squeeze(0).cpu().numpy()

    n_select = max(5, int(len(scores_v) * SELECTION_RATIO))
    thresh_v = np.percentile(scores_v, 100 - SELECTION_RATIO * 100)
    sel_v = sorted(np.where(scores_v >= thresh_v)[0])[:n_select]
    return scores_v, np.array(sel_v), thresh_v

# Build dataframe safely
rl_metrics_df = pd.DataFrame([
    {"Setting": "WITHOUT RL (Mamba Only)", **mamba_only_metrics},
    {"Setting": "WITH RL (Mamba + RL)", **mamba_rl_metrics},
])

print("📊 RL Contribution — Metrics Comparison:")
print(rl_metrics_df.to_string(index=False))

if 'DIRS' in globals() and 'outputs' in DIRS:
    rl_metrics_df.to_csv(f"{DIRS['outputs']}/rl_contribution_metrics.csv", index=False)

# ---------------------------------------------------------------------------
# 3. Safe model evaluation and frame comparison
# ---------------------------------------------------------------------------
m_norl = globals().get('model_mamba_only')
m_rl = globals().get('model')

scores_norl, sel_norl, thresh_norl = get_scores_and_selection(m_norl, use_clip_fusion=False)
scores_rl, sel_rl, thresh_rl = get_scores_and_selection(m_rl, use_clip_fusion=False)

set_norl, set_rl = set(sel_norl.tolist()), set(sel_rl.tolist())
union, intersection = set_norl | set_rl, set_norl & set_rl
jaccard = len(intersection) / len(union) if union else 0.0

print(f"\n🔍 Frame-selection comparison (demo video, ResNet50-only front-end for both, so the "
      f"ONLY difference is RL):")
print(f"   WITHOUT RL selected : {len(sel_norl)} frames")
print(f"   WITH RL selected    : {len(sel_rl)} frames")
print(f"   Overlapping frames  : {len(intersection)}")
print(f"   Jaccard similarity  : {jaccard:.3f}  (1.0 = identical selection, 0.0 = no overlap)")

# ---------------------------------------------------------------------------
# 4. Visualization & Reporting
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, scores_x, sel_x, thresh_x, title in [
    (axes[0], scores_norl, sel_norl, thresh_norl, "WITHOUT RL (Mamba Only)"),
    (axes[1], scores_rl, sel_rl, thresh_rl, "WITH RL (Mamba + RL)"),
]:
    x = range(len(scores_x))
    ax.fill_between(x, scores_x, alpha=0.3, color='royalblue')
    ax.plot(x, scores_x, color='royalblue', linewidth=1.2)
    ax.axhline(thresh_x, color='red', linestyle='--', linewidth=1.2)
    if len(sel_x) > 0:
        ax.scatter(sel_x, scores_x[sel_x], color='red', zorder=5, s=25)
    ax.set_title(f"Frame Importance — {title}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Frame Index'); ax.set_ylabel('Score'); ax.grid(True, alpha=0.3)

plt.tight_layout()
if 'DIRS' in globals() and 'outputs' in DIRS:
    plt.savefig(f"{DIRS['outputs']}/rl_contribution_frame_selection.png", dpi=150, bbox_inches='tight')
plt.show()

# Extract F1 scores safely with fallback defaults
f1_rl = mamba_rl_metrics.get('f1', mamba_rl_metrics.get('f1_score', 0.0))
f1_norl = mamba_only_metrics.get('f1', mamba_only_metrics.get('f1_score', 0.0))

f1_delta = f1_rl - f1_norl
f1_base = max(f1_norl, 1e-6)

rl_observation = (
    f"Adding RL {'improved' if f1_delta >= 0 else 'reduced'} TVSum test F-score by "
    f"{abs(f1_delta):.4f} ({abs(f1_delta)/f1_base*100:.1f}% relative to the Mamba-Only baseline). "
    f"On the demo video, RL changed {len(union) - len(intersection)} of {len(sel_rl)} selected "
    f"keyframes relative to the non-RL selection (Jaccard overlap = {jaccard:.2f})."
)
print("\n📝 Report-ready observation:")
print("  ", rl_observation)
print("\n⚠️  If TVSum used the synthetic fallback this session, treat the F-score delta above as")
print("   noise-level, not evidence of a real RL effect.")

## Cell 41 – NEW (Improvement 3): Narrative Reward Contribution Analysis — WITHOUT vs. WITH

Compares `model` (Mamba+RL, WITHOUT the narrative reward) against `model_mamba_rl_narrative`
(Mamba+RL+Narrative Reward, WITH it). Reuses the narratives and BLEU/ROUGE-L already
computed in Cell 38 — `"Mamba + RL"` there already **is** the WITHOUT baseline the
narrative-divergence scores were measured against, so nothing needs recomputing.

In [ ]:
without_reward_narrative = narratives_by_variant["Mamba + RL"]
with_reward_narrative = narratives_by_variant["Mamba + RL + Narrative Reward"]
bleu_val = narrative_divergence["Mamba + RL + Narrative Reward"]["bleu_vs_main"]
rouge_val = narrative_divergence["Mamba + RL + Narrative Reward"]["rougeL_vs_main"]

print("=" * 70)
print("📝 WITHOUT Narrative Reward (Mamba + RL):")
print("=" * 70)
print(without_reward_narrative)

print("\n" + "=" * 70)
print("📝 WITH Narrative Reward (Mamba + RL + Narrative Reward):")
print("=" * 70)
print(with_reward_narrative)

print(f"\n📊 BLEU (vs. WITHOUT, as reference)  : {bleu_val}")
print(f"📊 ROUGE-L (vs. WITHOUT, as reference): {rouge_val}")
print("   (Divergence measure, not an absolute quality score — see Cell 38's caveat.)")

# --- Lightweight, honestly-labeled proxy comparison (no ground-truth reference exists) ---
def text_stats(text):
    words = text.split()
    return {"word_count": len(words), "unique_word_ratio": round(len(set(w.lower() for w in words)) / max(len(words), 1), 3)}

stats_without = text_stats(without_reward_narrative)
stats_with = text_stats(with_reward_narrative)
event_quality_df = pd.DataFrame([
    {"Setting": "WITHOUT Narrative Reward", **stats_without},
    {"Setting": "WITH Narrative Reward", **stats_with},
])
print("\n📊 Narrative surface-level stats (proxy only — NOT a validated quality metric):")
print(event_quality_df.to_string(index=False))
event_quality_df.to_csv(f"{DIRS['outputs']}/narrative_reward_contribution.csv", index=False)

# --- If Cell 25 already ran this session, show its coherence breakdown for extra context ---
if 'reward_breakdown' in dir():
    print("\n📊 Coherence component breakdown from Cell 25 (single-video LLM-guided reward):")
    for k in ['coherence_llm', 'coherence_clip_consecutive_events', 'coherence_blend']:
        if k in reward_breakdown:
            print(f"   {k:32s}: {reward_breakdown[k]:.4f}")

print("\n📝 Report-ready observation:")
print(f"   Enabling the narrative reward changed the generated summary from {stats_without['word_count']} to "
      f"{stats_with['word_count']} words, with a BLEU/ROUGE-L divergence of {bleu_val:.2f}/{rouge_val:.4f} "
      f"from the non-narrative-reward baseline — indicating the reward measurably alters the narrative "
      f"content, though this alone doesn't prove the change is a quality improvement without a human or "
      f"reference-based judgment.")


## Cell 42 – Correct Project Methodology & Documentation (Improvement 5 — Documentation Only, No Code Changed)

**Why this cell exists:** your training logs show `Total Loss = Supervised Loss + RL Loss`
— the supervised term (`BCE` against TVSum/SumMe ground-truth importance scores) is real
and dominant in the loss. That makes this a **hybrid supervised + reinforcement-learning
framework**, not a purely unsupervised one, regardless of how the project title reads. The
training code is correct and unchanged — only how you *describe* it needs to match what it
actually does.

### Correct Project Description
*Mamba-Narrative is a hybrid supervised-and-reinforcement-learning framework for video
summarization. A selective state-space model (Mamba) scores frame importance, trained
jointly via (1) supervised learning against human-annotated importance scores from
TVSum/SumMe, and (2) an Actor-Critic reinforcement-learning objective that further shapes
frame selection using representativeness, diversity, coverage, and LLM-judged narrative
coherence. A sequence-aware video-language model (LLaVA-NeXT-Video) then generates
event-level descriptions of the selected keyframes, which are composed into a final
narrative summary.*

### Correct Methodology Explanation (for your report's Methodology section)
- State plainly that the frame-scoring network is trained with a **combined loss**:
  supervised binary cross-entropy against ground-truth importance labels, plus a
  reinforcement-learning policy-gradient term weighted by `λ_RL`.
- Describe the RL component accurately as **RL-based refinement layered on top of
  supervision**, not as the sole training signal.
- The LLM-guided narrative reward operates in two places, and your report should say so
  explicitly: (a) as a proxy signal during full-dataset training (feature-space coherence,
  since TVSum/SumMe provide no source frames for real captioning), and (b) as a real,
  caption-based signal during the short single-video policy-gradient refinement step,
  where actual LLaVA-NeXT-Video captions are available and computationally feasible.
- If your project's ablation results (Cells 36-41) show measurable deltas between the
  Mamba-Only, +RL, and +Narrative-Reward variants, cite those numbers directly as evidence
  for each component's contribution — don't rely on the architecture diagram alone to make
  the case.

### Correct Report Wording (drop-in replacements)
| Instead of | Write |
|---|---|
| "A fully unsupervised video summarization framework" | "A hybrid supervised and reinforcement-learning video summarization framework" |
| "The RL agent learns entirely from reward signals" | "The RL component refines a supervised base policy using representativeness, diversity, coverage, and narrative-coherence rewards" |
| "LLM-guided reward drives training" | "An LLM-guided narrative reward contributes to both dataset-level training (via a coherence proxy) and single-video refinement (via real captions)" |

### Correct Viva Explanation (what to say if asked "isn't this supervised, not unsupervised?")
*"You're right that the loss includes a supervised term — we use TVSum/SumMe's
human-annotated importance scores as one training signal, combined with a reinforcement-
learning objective. We describe it as a hybrid framework for that reason. The RL and
narrative-reward components are what's novel here: they let the model refine frame
selection using signals — representativeness, diversity, timeline coverage, and narrative
coherence from a video-language model — that the supervised labels alone don't capture,
and our ablation study (Cells 36-41) quantifies each component's contribution separately."*

This is a defensible, accurate answer — it doesn't overclaim "unsupervised," and it turns
the hybrid design into a strength (multiple complementary signals) rather than something to
downplay.


## Cell 32 – Save Model Checkpoint
*(MODIFIED — added a few lines to also save the SumMe-trained checkpoint; original TVSum checkpoint logic untouched)*

In [ ]:
import os
import json
import torch

# ---------------------------------------------------------------------------
# 1. Safe Hyperparameter & Variable Fallbacks
# ---------------------------------------------------------------------------
_num_epochs = globals().get('NUM_EPOCHS', 10)
_feature_dim = globals().get('FEATURE_DIM', 512)
_learning_rate = globals().get('LEARNING_RATE', 1e-4)
_lambda_rl = globals().get('LAMBDA_RL', globals().get('LAMBDA_REWARD', 0.1))
_weight_decay = globals().get('WEIGHT_DECAY', 1e-4)
_train_ratio = globals().get('TRAIN_RATIO', 0.8)

_checkpoint_path = globals().get('CHECKPOINT_PATH', './checkpoint.pt')
_checkpoint_summe_path = globals().get('CHECKPOINT_SUMME_PATH', './checkpoint_summe.pt')
_comparison_csv_path = globals().get('comparison_csv_path', './comparison.csv')
_base_dir = globals().get('BASE_DIR', '')

# ---------------------------------------------------------------------------
# 2. Build & Save Final Main Checkpoint
# ---------------------------------------------------------------------------
final_checkpoint = {
    'epoch': _num_epochs,
    'model_state': model.state_dict() if 'model' in globals() and model is not None else None,
    'critic_state': critic.state_dict() if 'critic' in globals() and critic is not None else None,
    'adapter_state': adapter.state_dict() if 'adapter' in globals() and adapter is not None else None,
    'optimizer_actor_state': optimizer_actor.state_dict() if 'optimizer_actor' in globals() else None,
    'optimizer_critic_state': optimizer_critic.state_dict() if 'optimizer_critic' in globals() else None,
    'scheduler_actor_state': scheduler_actor.state_dict() if 'scheduler_actor' in globals() else None,
    'scheduler_critic_state': scheduler_critic.state_dict() if 'scheduler_critic' in globals() else None,
    'history': globals().get('history', {}),
    'config': {
        'feat_dim': _feature_dim,
        'd_model': 256,
        'n_layers': 4,
        'd_state': 16,
        'lr': _learning_rate,
        'epochs': _num_epochs,
        'lambda_rl': _lambda_rl,
        'weight_decay': _weight_decay,
        'optimizer': 'AdamW',
        'train_ratio': _train_ratio,
    },
    'tvsum_metrics': globals().get('tvsum_metrics', {}),
    'summe_metrics': globals().get('summe_metrics', {}),
}

# Ensure destination folder exists
os.makedirs(os.path.dirname(os.path.abspath(_checkpoint_path)), exist_ok=True)
torch.save(final_checkpoint, _checkpoint_path)
print(f"✅ Model checkpoint saved: {_checkpoint_path}")
if os.path.exists(_checkpoint_path):
    print(f"   Size: {os.path.getsize(_checkpoint_path)/1e6:.2f} MB")

# ---------------------------------------------------------------------------
# 3. Persist SumMe Model Checkpoint & Artifact Checks
# ---------------------------------------------------------------------------
if 'model_summe' in globals() and model_summe is not None:
    summe_checkpoint = {
        'epoch': globals().get('NUM_EPOCHS_SUMME'),
        'model_state': model_summe.state_dict(),
        'critic_state': critic_summe.state_dict() if 'critic_summe' in globals() and critic_summe is not None else None,
        'history': globals().get('history_summe'),
    }
    os.makedirs(os.path.dirname(os.path.abspath(_checkpoint_summe_path)), exist_ok=True)
    torch.save(summe_checkpoint, _checkpoint_summe_path)
    print(f"✅ SumMe model checkpoint saved: {_checkpoint_summe_path}")

if 'comparison_df' in globals():
    print(f"✅ 3-experiment comparison table already saved: {_comparison_csv_path}")

if 'generalization_results' in globals() and generalization_results:
    gen_path = f"{DIRS['outputs']}/generalization_results.csv" if 'DIRS' in globals() and 'outputs' in DIRS else "./generalization_results.csv"
    print(f"✅ Generalization test results already saved: {gen_path}")

# ---------------------------------------------------------------------------
# 4. JSON Metrics Persistence
# ---------------------------------------------------------------------------
all_frames_count = len(globals().get('all_frames', []))
selected_indices_count = len(globals().get('selected_indices', []))
orig_dur_val = globals().get('orig_dur', 1.0)
summ_dur_val = globals().get('summ_dur', 0.0)

metrics_dict = {
    'tvsum_metrics': globals().get('tvsum_metrics', {}),
    'summe_metrics': globals().get('summe_metrics', {}),
    'video_info': {
        'filename': globals().get('uploaded_filename', 'unknown'),
        'total_frames': all_frames_count,
        'selected_frames': selected_indices_count,
        'selection_ratio_pct': (selected_indices_count / all_frames_count * 100) if all_frames_count > 0 else 0.0,
        'original_duration_s': orig_dur_val,
        'summary_duration_s': summ_dur_val,
        'compression_pct': ((1 - summ_dur_val / orig_dur_val) * 100) if orig_dur_val > 0 else 0.0,
    },
}

output_dir = DIRS['outputs'] if 'DIRS' in globals() and 'outputs' in DIRS else "."
os.makedirs(output_dir, exist_ok=True)
metrics_path = f"{output_dir}/metrics.json"

with open(metrics_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f"✅ Metrics saved: {metrics_path}")

# ---------------------------------------------------------------------------
# 5. Output Summary Report
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("📁 ALL OUTPUT FILES")
print("=" * 60)

summary_vid = globals().get('SUMMARY_VIDEO_PATH', './summary.mp4')
logs_dir = DIRS['logs'] if 'DIRS' in globals() and 'logs' in DIRS else "."

all_outputs = [
    (summary_vid, "🎬 Summary Video"),
    (_checkpoint_path, "🤖 Model Checkpoint"),
    (metrics_path, "📊 Evaluation Metrics"),
    (_comparison_csv_path, "📊 TVSum vs SumMe Comparison"),
    (f"{logs_dir}/training_curves.png", "📈 Training Curves"),
    (f"{output_dir}/keyframes_with_captions.png", "🖼️  Keyframes Grid"),
]

for path, label in all_outputs:
    if os.path.exists(path):
        display_path = path.replace(_base_dir, '...') if _base_dir else path
        print(f"  {label:32s} → {display_path} ({os.path.getsize(path)/1e6:.2f} MB)")
    else:
        print(f"  {label:32s} → ⚠️  Not found")

kf_dir = DIRS['keyframes'] if 'DIRS' in globals() and 'keyframes' in DIRS else None
if kf_dir and os.path.exists(kf_dir):
    kf_count = len([f for f in os.listdir(kf_dir) if f.endswith('.jpg')])
    display_kf_dir = kf_dir.replace(_base_dir, '...') if _base_dir else kf_dir
    print(f"  🖼️  Keyframes ({kf_count} images)              → {display_kf_dir}")

print("\n✅ All available outputs processed successfully!")

## Cell 33 – Generate Final Dashboard
A single-glance dashboard combining training curves, evaluation metrics on both datasets, frame
importance, and sample keyframes — suitable for the guide demonstration / viva.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(22, 16))
fig.suptitle('Mamba-Narrative: Final Project Dashboard', fontsize=18, fontweight='bold', y=0.98)
gs = fig.add_gridspec(4, 4, hspace=0.5, wspace=0.35)

# Safe scope getters for data sources
hist = globals().get('history', {})
tvsum_m = globals().get('tvsum_metrics', {})
summe_m = globals().get('summe_metrics', {})

# ---------------------------------------------------------------------------
# Plot 1: Training Loss Curve
# ---------------------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, :2])
if isinstance(hist, dict) and len(hist.get('epoch', [])) > 0:
    ax1.plot(hist['epoch'], hist['loss'], label='Total Loss', color='navy', lw=2)

    # Check for alternate keys for supervised loss (e.g., 'sup_loss', 'actor_loss')
    sup_loss_key = 'sup_loss' if 'sup_loss' in hist else ('actor_loss' if 'actor_loss' in hist else None)
    if sup_loss_key:
        ax1.plot(hist['epoch'], hist[sup_loss_key], label='Supervised', color='royalblue', lw=2, ls='--')

    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, "Training history not available\n(model loaded from checkpoint)", ha='center', va='center',
             fontsize=11, bbox=dict(facecolor='lightyellow', edgecolor='gray'))
    ax1.set_xticks([]); ax1.set_yticks([])
ax1.set_title('Training Loss Curve', fontweight='bold')

# ---------------------------------------------------------------------------
# Plot 2: RL Reward Progress
# ---------------------------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 2:])
if isinstance(hist, dict) and len(hist.get('epoch', [])) > 0 and 'reward' in hist:
    ax2.plot(hist['epoch'], hist['reward'], color='darkorange', lw=2, marker='o', ms=3)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Reward'); ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "RL reward history not available", ha='center', va='center', fontsize=11,
             bbox=dict(facecolor='lightyellow', edgecolor='gray'))
    ax2.set_xticks([]); ax2.set_yticks([])
ax2.set_title('RL Reward Progress', fontweight='bold')

# ---------------------------------------------------------------------------
# Plot 3a: TVSum Metrics
# ---------------------------------------------------------------------------
ax3 = fig.add_subplot(gs[1, :1])
names3 = ['Precision', 'Recall', 'F1', 'Diversity']
vals3 = [
    tvsum_m.get('precision', 0.0),
    tvsum_m.get('recall', 0.0),
    tvsum_m.get('f1', tvsum_m.get('f1_score', 0.0)),
    tvsum_m.get('diversity', 0.0)
]
bars = ax3.bar(names3, vals3, color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'], edgecolor='white')
for bar, val in zip(bars, vals3):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.2f}', ha='center', fontsize=8)
ax3.set_ylim(0, 1.15); ax3.set_title('TVSum Metrics', fontweight='bold'); ax3.grid(True, alpha=0.3, axis='y')

# ---------------------------------------------------------------------------
# Plot 3b: SumMe Metrics
# ---------------------------------------------------------------------------
ax3b = fig.add_subplot(gs[1, 1:2])
names3b = ['MSE', 'Spearman', 'F1']
vals3b = [
    summe_m.get('mse', 0.0),
    summe_m.get('spearman', 0.0),
    summe_m.get('f1', summe_m.get('f1_score', 0.0))
]
bars = ax3b.bar(names3b, vals3b, color=['#E53935', '#3949AB', '#FF9800'], edgecolor='white')
for bar, val in zip(bars, vals3b):
    ax3b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.2f}', ha='center', fontsize=8)
ax3b.set_title('SumMe Metrics', fontweight='bold'); ax3b.grid(True, alpha=0.3, axis='y')

# ---------------------------------------------------------------------------
# Plot 4: Frame Importance Scores
# ---------------------------------------------------------------------------
ax4 = fig.add_subplot(gs[1, 2:])
imp_scores = globals().get('importance_scores', np.array([]))
sel_indices = globals().get('selected_indices', np.array([]))
thresh_val = globals().get('threshold', 0.0)

if len(imp_scores) > 0:
    x = range(len(imp_scores))
    ax4.fill_between(x, imp_scores, alpha=0.3, color='royalblue')
    ax4.plot(x, imp_scores, color='royalblue', lw=1)
    if len(sel_indices) > 0:
        ax4.scatter(sel_indices, imp_scores[sel_indices], color='red', s=30, zorder=5, label='Selected')
    ax4.axhline(thresh_val, color='red', ls='--', lw=1.5)
    ax4.legend(fontsize=8)

ax4.set_title('Frame Importance Scores', fontweight='bold'); ax4.set_xlabel('Frame Index')
ax4.grid(True, alpha=0.3)

# ---------------------------------------------------------------------------
# Plot 5: Duration Comparison
# ---------------------------------------------------------------------------
ax5 = fig.add_subplot(gs[2, :1])
o_dur = globals().get('orig_dur', 0.0)
s_dur = globals().get('summ_dur', 0.0)

summary_stats = {'Original\n(s)': o_dur, 'Summary\n(s)': s_dur}
ax5.bar(summary_stats.keys(), summary_stats.values(), color=['#1976D2', '#43A047'], edgecolor='white', width=0.4)
for k, v in summary_stats.items():
    ax5.text(list(summary_stats.keys()).index(k), v + 0.5, f'{v:.1f}s', ha='center', fontsize=10, fontweight='bold')
ax5.set_title('Duration Comparison', fontweight='bold'); ax5.set_ylabel('Duration (s)'); ax5.grid(True, alpha=0.3, axis='y')

# ---------------------------------------------------------------------------
# Plot 6: Dataset Comparison
# ---------------------------------------------------------------------------
ax6 = fig.add_subplot(gs[2, 1:])
comp_df = globals().get('comparison_df')
if comp_df is not None and "TVSum" in comp_df and "SumMe" in comp_df:
    shared = comp_df.dropna(subset=["TVSum", "SumMe"], how='any')
    if len(shared) > 0:
        xpos = np.arange(len(shared)); w = 0.35
        ax6.bar(xpos - w/2, shared["TVSum"], w, label='TVSum', color='#1976D2')
        ax6.bar(xpos + w/2, shared["SumMe"], w, label='SumMe', color='#E53935')
        ax6.set_xticks(xpos); ax6.set_xticklabels(shared["Metric"], fontsize=8)
        ax6.legend(fontsize=8)
ax6.set_title('TVSum vs SumMe Comparison', fontweight='bold'); ax6.grid(True, alpha=0.3, axis='y')

# ---------------------------------------------------------------------------
# Row 4: Keyframe Grids
# ---------------------------------------------------------------------------
kf_imgs = globals().get('keyframe_images', [])
kf_caps = globals().get('keyframe_captions', [])
n_show_kf = min(4, len(kf_imgs))

for col in range(4):
    ax = fig.add_subplot(gs[3, col])
    if col < n_show_kf:
        ax.imshow(kf_imgs[col])
        cap_text = kf_caps[col] if col < len(kf_caps) else ""
        cap_short = cap_text[:40] + '...' if len(cap_text) > 40 else cap_text
        ax.set_title(f"KF {col+1}: {cap_short}", fontsize=7)
    else:
        ax.text(0.5, 0.5, "N/A", ha='center', va='center', fontsize=10, color='gray')
    ax.axis('off')

plt.tight_layout()

# Safe saving using DIRS if defined
if 'DIRS' in globals() and 'outputs' in DIRS:
    out_path = f"{DIRS['outputs']}/final_dashboard.png"
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f"✅ Final dashboard saved to {out_path}!")
else:
    plt.savefig("./final_dashboard.png", dpi=150, bbox_inches='tight')
    print("✅ Final dashboard saved to ./final_dashboard.png!")

plt.show()

## Cell 34 – Demo Results
A single, guide/viva-ready printout of the project status and headline results.

In [ ]:
from datetime import datetime

# Safe variable accessors
_history = globals().get('history', {})
_checkpoint_path = globals().get('CHECKPOINT_PATH', 'N/A')
_summary_video_path = globals().get('SUMMARY_VIDEO_PATH', 'N/A')
_tvsum_metrics = globals().get('tvsum_metrics', {})
_summe_metrics = globals().get('summe_metrics', {})
_narrative_summary = globals().get('narrative_summary', '[Narrative summary unavailable]')

# Safely extract TVSum metrics
tv_f1 = _tvsum_metrics.get('f1', _tvsum_metrics.get('f1_score', 0.0))
tv_prec = _tvsum_metrics.get('precision', 0.0)
tv_rec = _tvsum_metrics.get('recall', 0.0)

# Safely extract SumMe metrics with fallback checking
summe_f1 = _summe_metrics.get('f1', _summe_metrics.get('f1_score', 0.0))
summe_spearman = _summe_metrics.get('spearman', _summe_metrics.get('rho', 0.0))
summe_mse_val = _summe_metrics.get('mse', _summe_metrics.get('mean_squared_error'))
summe_mse_str = f"{summe_mse_val:.4f}" if isinstance(summe_mse_val, (int, float)) else "N/A"

print("=" * 70)
print("   MAMBA-NARRATIVE — DEMO RESULTS")
print("   Unsupervised Video Summarization via Selective State Space")
print("   Modelling and LLM-Guided Reinforcement Learning")
print("=" * 70)
print(f"  Run timestamp        : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Dataset(s) used       : TVSum (train/test) + SumMe (cross-dataset eval)")
print(f"  Training status       : {'Completed' if isinstance(_history, dict) and len(_history.get('epoch', [])) > 0 else 'Loaded from checkpoint'}")
print(f"  Checkpoint path       : {_checkpoint_path}")
print(f"  Summary video path    : {_summary_video_path}")
print("-" * 70)
print("  QUANTITATIVE METRICS")
print(f"    TVSum  — F1: {tv_f1:.4f} | Precision: {tv_prec:.4f} | Recall: {tv_rec:.4f}")
print(f"    SumMe  — MSE: {summe_mse_str} | F1: {summe_f1:.4f} | Spearman: {summe_spearman:.4f}")
print("-" * 70)
print("  LLM-GENERATED NARRATIVE SUMMARY")
print(f"    {_narrative_summary}")
print("=" * 70)
print("  ✅ READY FOR DEMONSTRATION")
print("=" * 70)

## Cell 35 – Architecture Diagram (Graphviz)
An auto-generated diagram of the full pipeline, including the LLM-guided feedback loop into the RL
reward, for inclusion in the report/slides.

In [ ]:
dot = graphviz.Digraph('MambaNarrativePipeline', format='png')
dot.attr(rankdir='TB', bgcolor='white', fontsize='12')
dot.attr('node', shape='box', style='rounded,filled', fontname='Helvetica', fontsize='11')

dot.node('video',     'Video Input',                                    fillcolor='#E3F2FD')
dot.node('feat',      'Feature Extraction\n(ResNet50 / GoogLeNet pool5)', fillcolor='#BBDEFB')
dot.node('ssm',       'Mamba Network\n(Selective State Space)',          fillcolor='#90CAF9')
dot.node('llava',     'LLaVA-NeXT-Video\n(Event Description)',           fillcolor='#FFE082')
dot.node('flant5',    'Flan-T5\n(Narrative Generation)',                 fillcolor='#FFCC80')
dot.node('ranking',   'LLM-based\nFrame Ranking',                        fillcolor='#FFB74D')
dot.node('coherence', 'LLM Narrative\nCoherence Scoring',                fillcolor='#FFA726')
dot.node('reward',    'Reinforcement Learning\nReward',                  fillcolor='#42A5F5')
dot.node('pg',        'Policy Gradient\nOptimization',                   fillcolor='#64B5F6')
dot.node('keyframes', 'Final Keyframe\nSelection',                       fillcolor='#FFF9C4')
dot.node('summary',   'Summary Video',                                   fillcolor='#C8E6C9')
dot.node('eval',      'Evaluation\n(TVSum + SumMe)',                     fillcolor='#A5D6A7')
dot.node('dash',      'Dashboard',                                       fillcolor='#81C784')

for a, b in [('video', 'feat'), ('feat', 'ssm'), ('ssm', 'llava'), ('llava', 'flant5'),
             ('flant5', 'ranking'), ('ranking', 'coherence'), ('coherence', 'reward'),
             ('reward', 'pg'), ('pg', 'keyframes'), ('keyframes', 'summary'),
             ('summary', 'eval'), ('eval', 'dash')]:
    dot.edge(a, b)

dot.edge('pg', 'ssm', label='refines policy', style='dashed', color='red', fontcolor='red')

diagram_path = dot.render(f"{DIRS['outputs']}/pipeline_architecture", cleanup=True)
print(f"✅ Architecture diagram saved to: {diagram_path}")
dot

In [ ]:
# =============================================================================
# MAMBA-NARRATIVE — INTERACTIVE DEMO UI (Gradio)
# =============================================================================
# Paste this as the LAST cell in your notebook. Run it AFTER all other cells
# have executed (training loaded, evaluation done, etc.).
#
# It creates a tabbed web UI with:
#   Tab 1: Project Overview & Architecture
#   Tab 2: Upload Video → Full Summarization Pipeline
#   Tab 3: Training Curves & Reward Analysis
#   Tab 4: TVSum / SumMe Evaluation Results
#   Tab 5: Ablation Study Comparison
#   Tab 6: Efficiency & Model Info
#
# A public share link is generated automatically — open it on any device.
# =============================================================================

!pip install -q gradio>=4.0

import gradio as gr
import tempfile
import traceback

# ---- Helper: safely read variables from the notebook scope ----
def safe_get(name, default=None):
    return globals().get(name, default)


# =========================================================================
# TAB 1: PROJECT OVERVIEW
# =========================================================================
def build_overview_tab():
    overview_md = """
# Mamba-Narrative
### Unsupervised Video Summarization via Selective State Space Modeling and Narrative-Coherent Reinforcement Learning

---

### Pipeline
```
Raw Video → Feature Extraction → Mamba SSM → REINFORCE Policy
    → Unsupervised Reward (Repr + Diversity + Coverage + Coherence + Length)
    → Actor-Critic Update → Keyframe Selection
    → [Demo] LLaVA-NeXT Captioning → Flan-T5 Narrative
```

### Key Contributions
1. **Mamba SSM** for O(n) linear-time temporal modeling (vs. Transformer O(n²))
2. **Fully unsupervised** training — no human labels in the training loop
3. **Narrative coherence proxy** reward for story-aware keyframe selection
4. **5-fold cross-validation** with knapsack-based F-score (standard protocol)

### Training Mode
- **Unsupervised RL** (REINFORCE with Bernoulli action sampling)
- Human importance scores used **only** at evaluation time
- Reward: Representativeness (0.30) + Length (0.25) + Diversity (0.15) + Coverage (0.15) + Coherence (0.15)
"""
    return gr.Markdown(overview_md)


# =========================================================================
# TAB 2: VIDEO SUMMARIZATION DEMO
# =========================================================================
def summarize_video(video_file, selection_pct, progress=gr.Progress()):
    if video_file is None:
        return None, "Please upload a video file.", None, None, None

    try:
        progress(0.1, desc="Extracting frames...")

        extractor_cls = safe_get('VideoFeatureExtractor')
        if extractor_cls is None:
            return None, "VideoFeatureExtractor not found. Run Cell 16 first.", None, None, None

        extractor = extractor_cls(device)
        features_np, frames_pil, fps = extractor.extract_from_video(video_file)
        frames = [np.array(img) for img in frames_pil]

        if len(frames) == 0:
            return None, "No frames extracted from video.", None, None, None

        progress(0.4, desc=f"Extracted {len(frames)} frames. Running Mamba model...")
                # Use the EXACT same pipeline as Cell 43+47:
        # ResNet(2048) + CLIP(512) → FusionModule(2560) → Adapter(2560→1024) → Mamba
        _clip_extractor = safe_get('clip_extractor')
        _fusion_module = safe_get('fusion_module')
        _adapter = safe_get('adapter')

        if _clip_extractor is not None and _fusion_module is not None and _adapter is not None:
            clip_features = _clip_extractor.extract_from_frames(frames_pil)
            fused_features = _fusion_module.forward_np(features_np, clip_features)
            fused_features = sk_normalize(fused_features, norm='l2')
            raw_features = torch.FloatTensor(fused_features).to(device)
            with torch.no_grad():
                raw_features = _adapter(raw_features)
        else:
            raw_features = torch.FloatTensor(features_np).to(device)
            if raw_features.shape[-1] != FEATURE_DIM:
                with torch.no_grad():
                    proj = nn.Linear(raw_features.shape[-1], FEATURE_DIM).to(device)
                    raw_features = proj(raw_features)

        # Run model inference
        _model = safe_get('model')
        if _model is None:
            return None, "Model not loaded. Run Cell 26-30 first.", None, None, None

        _model.eval()
        with torch.no_grad():
            if raw_features.dim() == 2:
                raw_features = raw_features.unsqueeze(0)
            mask = torch.ones(1, raw_features.shape[1], device=device)
            scores = _model(raw_features, mask).squeeze(0).cpu().numpy()

        progress(0.6, desc="Selecting keyframes...")

        # Select frames based on percentage
        n_select = max(1, int(len(scores) * selection_pct / 100))
        top_indices = np.argsort(scores)[::-1][:n_select]
        top_indices = sorted(top_indices)
        threshold = scores[top_indices[-1]] if len(top_indices) > 0 else 0.5

        # Build importance plot
        import matplotlib
        matplotlib.use('Agg')
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(scores, color='#2196F3', linewidth=0.8, alpha=0.8)
        ax.fill_between(range(len(scores)), scores, alpha=0.2, color='#2196F3')
        for idx in top_indices:
            ax.axvline(x=idx, color='#FF5722', alpha=0.4, linewidth=0.5)
        ax.axhline(y=threshold, color='red', linestyle='--', alpha=0.5,
                   label=f'Threshold={threshold:.3f}')
        ax.set_xlabel('Frame Index'); ax.set_ylabel('Importance Score')
        ax.set_title(f'Frame Importance Scores ({len(top_indices)} keyframes selected)')
        ax.legend(); ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plot_path = tempfile.mktemp(suffix='.png')
        fig.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

        progress(0.75, desc="Building keyframe gallery...")

        # Keyframe gallery
        keyframe_images = []
        for idx in top_indices[:24]:
            img = Image.fromarray(frames[idx])
            keyframe_images.append((img, f"Frame {idx} (score: {scores[idx]:.3f})"))

        progress(0.85, desc="Generating summary video...")

        # Summary video
        summary_path = None
        try:
            create_summary = safe_get('create_summary_video')
            if create_summary is not None:
                summary_path = tempfile.mktemp(suffix='.mp4')
                create_summary(video_file, top_indices, fps, summary_path)
                if not os.path.exists(summary_path) or os.path.getsize(summary_path) < 1000:
                    summary_path = None
        except Exception:
            summary_path = None

        # Captions (if VLM is loaded)
        captions_text = ""
        try:
            get_captions = safe_get('get_captions_for_indices')
            vlm_backend = safe_get('VLM_BACKEND', 'disabled')
            if get_captions is not None and vlm_backend != 'disabled':
                progress(0.9, desc="Generating captions...")
                caps = get_captions(frames, list(top_indices[:10]))
                for i, (idx, cap) in enumerate(zip(top_indices[:10], caps)):
                    captions_text += f"**Frame {idx}**: {cap}\n\n"
            else:
                captions_text = "VLM not loaded — captions unavailable."
        except Exception as e:
            captions_text = f"Caption generation failed: {e}"

        # Narrative (if Flan-T5 loaded)
        narrative_text = ""
        try:
            llm_model = safe_get('llm_model')
            llm_tokenizer = safe_get('llm_tokenizer')
            if llm_model is not None and captions_text and "unavailable" not in captions_text:
                caps_list = [c.split(": ", 1)[1] if ": " in c else c
                             for c in captions_text.strip().split("\n\n") if c.strip()]
                prompt = ("Summarize this video based on these scene descriptions into a "
                          "coherent narrative paragraph:\n" + "\n".join(caps_list[:10]))
                inputs = llm_tokenizer(prompt, return_tensors="pt", max_length=512,
                                       truncation=True).to(device)
                with torch.no_grad():
                    out = llm_model.generate(**inputs, max_new_tokens=150, num_beams=2)
                narrative_text = llm_tokenizer.decode(out[0], skip_special_tokens=True)
            else:
                narrative_text = "Flan-T5 not loaded or no captions available."
        except Exception as e:
            narrative_text = f"Narrative generation failed: {e}"

        stats_text = (
            f"**Frames extracted:** {len(frames)}\n\n"
            f"**Keyframes selected:** {len(top_indices)} ({100*len(top_indices)/len(frames):.1f}%)\n\n"
            f"**Score range:** {scores.min():.4f} - {scores.max():.4f}\n\n"
            f"**Threshold:** {threshold:.4f}\n\n"
            f"---\n\n### Narrative Summary\n\n{narrative_text}"
        )

        return summary_path, stats_text, plot_path, keyframe_images, captions_text

    except Exception as e:
        tb = traceback.format_exc()
        return None, f"Error: {e}\n\n```\n{tb}\n```", None, None, None


def build_demo_tab():
    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label="Upload Video (MP4/AVI/MOV)")
            pct_slider = gr.Slider(5, 30, value=15, step=1,
                                   label="Selection % (target keyframe ratio)")
            run_btn = gr.Button("Summarize Video", variant="primary", size="lg")

        with gr.Column(scale=1):
            stats_output = gr.Markdown(label="Summary Statistics")

    with gr.Row():
        plot_output = gr.Image(label="Frame Importance Scores", height=250)

    with gr.Row():
        summary_video = gr.Video(label="Generated Summary Video")

    with gr.Row():
        gallery = gr.Gallery(label="Selected Keyframes", columns=6, height=300)

    with gr.Row():
        captions_output = gr.Markdown(label="Event Captions")

    run_btn.click(
        fn=summarize_video,
        inputs=[video_input, pct_slider],
        outputs=[summary_video, stats_output, plot_output, gallery, captions_output],
    )


# =========================================================================
# TAB 3: TRAINING CURVES
# =========================================================================
def generate_training_plots():
    _history = safe_get('history', {})
    if not _history or not _history.get('epoch'):
        return None, "No training history found. Run training first."

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Mamba-Narrative — Unsupervised RL Training', fontsize=15, fontweight='bold')
    epochs = _history['epoch']

    # Policy Loss
    if 'loss' in _history:
        axes[0, 0].plot(epochs, _history['loss'], 'b-', linewidth=1.5)
        axes[0, 0].set_title('Policy Loss'); axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].grid(True, alpha=0.3)

    # Total Reward
    if 'reward' in _history:
        axes[0, 1].plot(epochs, _history['reward'], 'g-', linewidth=2, label='Total')
        axes[0, 1].set_title('Total Reward'); axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    # Reward Components
    colors = {'r_repr': '#2196F3', 'r_div': '#9C27B0', 'r_cov': '#FF9800',
              'r_coh': '#E91E63', 'r_len': '#4CAF50'}
    for key, color in colors.items():
        if key in _history and len(_history[key]) > 0:
            label = key.replace('r_', '').capitalize()
            axes[1, 0].plot(epochs, _history[key], color=color, linewidth=1.2, label=label)
    axes[1, 0].set_title('Reward Components'); axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend(fontsize=8); axes[1, 0].grid(True, alpha=0.3)

    # Entropy
    if 'entropy' in _history and len(_history['entropy']) > 0:
        axes[1, 1].plot(epochs, _history['entropy'], 'r-', linewidth=1.5)
    axes[1, 1].set_title('Exploration (Entropy)'); axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = tempfile.mktemp(suffix='.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    summary = (
        f"**Epochs trained:** {len(epochs)}\n\n"
        f"**Final reward:** {_history['reward'][-1]:.4f}\n\n"
        f"**Final loss:** {_history['loss'][-1]:.4f}\n\n"
        f"**Training mode:** Unsupervised RL (no BCE)\n\n"
    )
    if 'entropy' in _history and len(_history['entropy']) > 0:
        summary += f"**Final entropy:** {_history['entropy'][-1]:.4f}"

    return path, summary


def build_training_tab():
    plot_btn = gr.Button("Generate Training Plots", variant="primary")
    with gr.Row():
        train_plot = gr.Image(label="Training Curves", height=500)
        train_stats = gr.Markdown()
    plot_btn.click(fn=generate_training_plots, outputs=[train_plot, train_stats])


# =========================================================================
# TAB 4: EVALUATION RESULTS
# =========================================================================
def get_evaluation_results():
    _tvsum = safe_get('tvsum_metrics', {})
    _summe = safe_get('summe_metrics', {})
    _synthetic = safe_get('USING_SYNTHETIC_DATA', False)

    if not _tvsum:
        return None, "No evaluation results. Run Cells 66-70 first."

    # Build comparison bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # TVSum
    if _tvsum:
        names = ['F-score', 'Precision', 'Recall', 'Spearman']
        vals = [_tvsum.get('f1', 0), _tvsum.get('precision', 0),
                _tvsum.get('recall', 0), _tvsum.get('spearman', 0)]
        bars = axes[0].bar(names, vals, color=['#FF9800', '#2196F3', '#4CAF50', '#9C27B0'],
                           edgecolor='white', width=0.6)
        for bar, val in zip(bars, vals):
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                         f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
        axes[0].set_ylim(0, 1.1); axes[0].set_title('TVSum (Knapsack F-score)', fontweight='bold')
        axes[0].grid(True, alpha=0.2, axis='y')

    # SumMe
    if _summe:
        names = ['F-score', 'Spearman', 'Compression']
        vals = [_summe.get('f1', 0), _summe.get('spearman', 0), _summe.get('compression_ratio', 0)]
        bars = axes[1].bar(names, vals, color=['#FF9800', '#9C27B0', '#607D8B'],
                           edgecolor='white', width=0.5)
        for bar, val in zip(bars, vals):
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                         f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
        axes[1].set_ylim(0, 1.1); axes[1].set_title('SumMe Cross-Dataset', fontweight='bold')
        axes[1].grid(True, alpha=0.2, axis='y')

    plt.tight_layout()
    path = tempfile.mktemp(suffix='.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Text summary
    md = "## Evaluation Results (Knapsack-based, Standard Protocol)\n\n"
    if _synthetic:
        md += "> **WARNING: Results are on SYNTHETIC data — not valid for publication.**\n\n"

    if _tvsum:
        md += "### TVSum (In-Domain)\n\n"
        md += f"| Metric | Value |\n|--------|-------|\n"
        for k, v in _tvsum.items():
            md += f"| {k} | {v:.4f} |\n" if isinstance(v, float) else f"| {k} | {v} |\n"

    if _summe:
        md += "\n### SumMe (Cross-Dataset Transfer)\n\n"
        md += f"| Metric | Value |\n|--------|-------|\n"
        for k, v in _summe.items():
            md += f"| {k} | {v:.4f} |\n" if isinstance(v, float) else f"| {k} | {v} |\n"

    # Published baselines for comparison
    md += "\n### Comparison with Published Unsupervised Methods (TVSum)\n\n"
    md += "| Method | F-score |\n|--------|--------|\n"
    md += "| Random | ~0.40 |\n"
    md += "| DR-DSN (2018) | ~0.57 |\n"
    md += "| SUM-GAN (2018) | ~0.56 |\n"
    md += "| CSNet (2019) | ~0.58 |\n"
    md += f"| **Mamba-Narrative (Ours)** | **{_tvsum.get('f1', 0):.3f}** |\n"

    return path, md


def build_eval_tab():
    eval_btn = gr.Button("Show Evaluation Results", variant="primary")
    with gr.Row():
        eval_plot = gr.Image(label="Evaluation Charts", height=350)
    eval_md = gr.Markdown()
    eval_btn.click(fn=get_evaluation_results, outputs=[eval_plot, eval_md])


# =========================================================================
# TAB 5: ABLATION STUDY
# =========================================================================
def get_ablation_results():
    _ablation = safe_get('ablation_metrics', {})
    if not _ablation:
        return None, "No ablation results. Run Cells 86-88 first."

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    names = list(_ablation.keys())
    f1_vals = [_ablation[n].get('f1', 0) for n in names]
    short_names = [n.replace('Mamba ', 'M').replace(' (basic)', '')
                   .replace(' + ', '+').replace('Full Pipeline (+CLIP demo)', 'Full+CLIP')
                   for n in names]

    colors = ['#9E9E9E', '#2196F3', '#FF9800', '#4CAF50']
    bars = ax.bar(short_names, f1_vals, color=colors[:len(names)], edgecolor='white', width=0.6)
    for bar, val in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('F-score (Knapsack)'); ax.set_title('Ablation Study — Component Contributions',
                                                        fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.2, axis='y'); ax.set_ylim(0, max(f1_vals) * 1.3 + 0.05)
    plt.tight_layout()
    path = tempfile.mktemp(suffix='.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    md = "## Ablation Study Results\n\n"
    md += "All variants trained for the **same number of epochs** with independent evaluation.\n\n"
    md += "| Variant | F-score | Spearman |\n|---------|---------|----------|\n"
    for name, m in _ablation.items():
        md += f"| {name} | {m.get('f1', 0):.4f} | {m.get('spearman', 0):.4f} |\n"
    md += "\n**Note:** Full Pipeline = same TVSum model as Mamba+RL+Coherence. "
    md += "CLIP fusion applies only to demo video at inference time."

    return path, md


def build_ablation_tab():
    abl_btn = gr.Button("Show Ablation Results", variant="primary")
    abl_plot = gr.Image(label="Ablation Comparison", height=400)
    abl_md = gr.Markdown()
    abl_btn.click(fn=get_ablation_results, outputs=[abl_plot, abl_md])


# =========================================================================
# TAB 6: MODEL INFO & EFFICIENCY
# =========================================================================
def get_model_info():
    _model = safe_get('model')
    if _model is None:
        return "Model not loaded."

    total_params = sum(p.numel() for p in _model.parameters())
    trainable_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)

    md = "## Model Architecture\n\n"
    md += f"| Property | Value |\n|----------|-------|\n"
    md += f"| Architecture | Mamba-Narrative (Selective SSM) |\n"
    md += f"| Total Parameters | {total_params:,} |\n"
    md += f"| Trainable Parameters | {trainable_params:,} |\n"
    md += f"| Input Feature Dim | {safe_get('FEATURE_DIM', 'N/A')} |\n"
    md += f"| Mamba d_model | 256 |\n"
    md += f"| Mamba d_state | 16 |\n"
    md += f"| Mamba layers | 4 |\n"
    md += f"| Training | Unsupervised RL (REINFORCE) |\n"
    md += f"| Complexity | O(n) linear time |\n"
    md += f"| Device | {device} |\n"

    md += "\n## Reward Function\n\n"
    md += "$$R_{total} = 0.30 \\cdot R_{repr} + 0.15 \\cdot R_{div} + 0.25 \\cdot R_{len} "
    md += "+ 0.15 \\cdot R_{cov} + 0.15 \\cdot R_{coh}$$\n\n"
    md += "All components are **unsupervised** — no human labels in training.\n\n"

    md += "## Mamba State Space Equations\n\n"
    md += "$$h_t = \\bar{A}_t \\cdot h_{t-1} + \\bar{B}_t \\cdot x_t$$\n"
    md += "$$y_t = C_t \\cdot h_t$$\n\n"
    md += "Where A, B, C are **input-dependent** (selective) matrices.\n"

    # Inference time if available
    md += "\n## Efficiency\n\n"
    md += "- Mamba SSM processes sequences in **O(n)** time\n"
    md += "- Transformers require **O(n²)** for the same task\n"
    md += "- Memory scales linearly with sequence length\n"

    return md


def build_info_tab():
    info_btn = gr.Button("Load Model Info", variant="primary")
    info_md = gr.Markdown()
    info_btn.click(fn=get_model_info, outputs=[info_md])


# =========================================================================
# ASSEMBLE THE APP
# =========================================================================
with gr.Blocks(
    title="Mamba-Narrative | Video Summarization",
    theme=gr.themes.Soft(primary_hue="orange", secondary_hue="blue"),
    css="""
    .gradio-container { max-width: 1200px !important; }
    h1 { text-align: center; }
    """
) as demo:

    gr.Markdown("# 🎬 Mamba-Narrative — Video Summarization Demo")
    gr.Markdown("*Unsupervised Video Summarization via Selective State Space Modeling "
                "and Narrative-Coherent Reinforcement Learning*")

    with gr.Tabs():
        with gr.Tab("📖 Overview"):
            build_overview_tab()

        with gr.Tab("🎥 Summarize Video"):
            build_demo_tab()

        with gr.Tab("📈 Training Curves"):
            build_training_tab()

        with gr.Tab("📊 Evaluation"):
            build_eval_tab()

        with gr.Tab("🔬 Ablation Study"):
            build_ablation_tab()

        with gr.Tab("⚙️ Model Info"):
            build_info_tab()

    gr.Markdown("---")
    gr.Markdown("*Mamba-Narrative | Final Year Project | Deep Learning & Computer Vision*")


# Launch with a public share link (works from Colab)
demo.launch(share=True, quiet=False)
